# SpanCall Lean Flow **v2.7** — Object-Param Decomposition

End-to-end training and step-by-step evaluation notebook for the **v2.7** change set. The pipeline is v2.6 (Hermes V1 + `Salesforce/xlam-function-calling-60k` 1-call subset, `answerdotai/ModernBERT-large`, `MAX_SEQ_LEN = 4096`); what changes is **how `object`-typed params are represented**:

- **Object decomposition.** A declared nested object (`{"filters": {"color": "red", "size": 10}}`) is split into scalar dotted pseudo-params (`filters.color`, `filters.size`) at the schema layer, aligned like any other scalar, and **reassembled into the nested dict by the resolver before casting, schema validation and the whole-call gates**. See `spancall/data/flatten.py` and `plans/lean_v2_7_object_decomposition_plan.md`.
- **Why.** In v2.6 an object value fell through the scalar branch of `ReverseAligner._align_value` and ended at `OFF_WINDOW`, i.e. it was silently *unresolved* — a coverage loss, and the reason object params never showed up in the FP dump. Decomposition converts them into the single-span selection the lean flow is good at.
- **What this notebook must answer.** (a) did coverage on object-bearing tools rise; (b) did exact-payload precision hold or improve; (c) how much of the residual error is still array-shaped.
- **Explicitly unchanged.** Arrays (element selection, cardinality, ordering) and per-param refusal. The v2.6 analysis showed arrays are the dominant precision source; that work is out of scope here and the audit in §4b keeps tracking it so it cannot hide.

**Path changes vs v2.6:** `data/tensorized_lean_v2_7` and `results_lean_v2_7`, and the tensors are rebuilt from `data/processed_v2_7` (realigned through the v2.7 template).

---
## 0. Setup & Configuration

Clones or updates the SpanCall repository, installs dependencies in editable mode, and configures v2.7 paths (`data/processed_v2_7`, `data/tensorized_lean_v2_7`, `results_lean_v2_7`). The data step below re-realigns through the v2.7 template because object decomposition changes the rendered param list.

In [1]:
# 0a. Pull latest code from public GitHub repository and install dependencies
import os, sys
from pathlib import Path

GIT_REPO_URL = "https://github.com/eleluong/spancall.git"
GIT_BRANCH   = "main"

cwd = Path.cwd().resolve()
if (cwd / "spancall" / "__init__.py").is_file():
    PROJECT_ROOT = cwd
elif (cwd.parent / "spancall" / "__init__.py").is_file():
    PROJECT_ROOT = cwd.parent
elif (cwd / "spancall" / "spancall" / "__init__.py").is_file():
    PROJECT_ROOT = cwd / "spancall"
else:
    print(f"Cloning public repository from {GIT_REPO_URL}...")
    !git clone --branch {GIT_BRANCH} {GIT_REPO_URL}
    PROJECT_ROOT = cwd / "spancall"

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if (PROJECT_ROOT / ".git").is_dir():
    print(f"Pulling latest code from {GIT_BRANCH}...")
    !git pull origin {GIT_BRANCH} || echo "git pull note: continuing with local files"

print(f"SpanCall project root: {PROJECT_ROOT}")
%pip install -q -e ".[model,data,agent,dev]" huggingface_hub

Pulling latest code from main...
From https://github.com/eleluong/spancall
 * branch            main       -> FETCH_HEAD
Already up to date.
SpanCall project root: /content/spancall
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for spancall (pyproject.toml) ... done


In [2]:
# 0b. Configuration & Environment (Full Data, v2.7 object decomposition)
import os, subprocess, sys
from pathlib import Path
import torch
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "spancall" / "__init__.py").is_file():
    if (PROJECT_ROOT.parent / "spancall" / "__init__.py").is_file():
        PROJECT_ROOT = PROJECT_ROOT.parent
    elif (PROJECT_ROOT / "spancall" / "spancall" / "__init__.py").is_file():
        PROJECT_ROOT = PROJECT_ROOT / "spancall"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from spancall.agent.backends.litellm_adapter import _load_dotenv_values

ENV = {**_load_dotenv_values(PROJECT_ROOT / ".env"), **os.environ}
HF_DATASET_ID = ENV.get("DATASET_NAME", "convoicon/spancall")
HF_TOKEN      = ENV.get("HUGGING_FACE_TOKEN") or ENV.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

DRIVE_MOUNT       = False
DRIVE_OUT         = "/content/drive/MyDrive/spancall_lean_artifacts"

DATA_DIR          = str(PROJECT_ROOT / "data" / "processed_v2_7")
V1_DATA_DIR       = str(PROJECT_ROOT / "data" / "processed")
TENSOR_DIR        = str(PROJECT_ROOT / "data" / "tensorized_lean_v2_7")
RESULTS_DIR       = str(PROJECT_ROOT / "results_lean_v2_7")
EVAL_SPLIT        = "hermes_eval"
TEST_SPLITS       = [
    "hermes_test_ood_schema",
    "hermes_test_ood_para",
    "hermes_test_adversarial",
    "xlam_eval",
    "xlam_test_ood_schema",
]

# xLAM-60k (1-call) mixing knobs:
#   0     -> Hermes-only baseline (Run A)
#   5000  -> Hermes + 5k xLAM (Run B)
#   15000 -> Hermes + 15k xLAM (Run C, default)
#   None  -> Hermes + full xLAM train
XLAM_TRAIN_ROWS   = 15000
XLAM_DEV_ROWS     = 1000
SINGLE_CALL_ONLY  = True

MAX_UTT_STALE_RATE = 0.02
MAX_UNRESOLVABLE   = 0.02
ALLOW_STALE_LABELS = False
FORCE_REALIGN      = False
FORCE_HF_DOWNLOAD  = False
MIN_JOINT_MARGIN   = 0.02

DEVICE          = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
EXPERIMENT_NAME = "spancall_lean_modernbert"
SEED            = 42

BACKBONE       = "answerdotai/ModernBERT-large"
MAX_SEQ_LEN    = 4096
MAX_SPAN_WIDTH = 16
MAX_CANDS      = 1024

_cfg_yaml = {}
try:
    with open("configs/default.yaml") as _fh:
        _cfg_yaml = yaml.safe_load(_fh) or {}
except Exception as _cfg_exc:
    print(f"[!] configs/default.yaml unreadable ({type(_cfg_exc).__name__}: {_cfg_exc})")

MAX_POS = int(((_cfg_yaml.get("optimization") or {}).get("max_pos", 3)) or 3)
_data_cfg = _cfg_yaml.get("data") or {}
MAX_TOOLS = _data_cfg.get("max_tools")
MAX_TOOLS = None if MAX_TOOLS in (None, "", 0) else int(MAX_TOOLS)
JITTER_PACK_TOOLS = bool(_data_cfg.get("jitter_pack_tools", True))

SPLIT_INPUTS = {
    "t2":  ["hermes_train.jsonl", "xlam_train.jsonl"],
    "dev": ["hermes_dev.jsonl", "xlam_dev.jsonl"],
}
SAMPLE_LIMITS     = {"t2": None, "dev": None}
DATA_FRACTION = 0.5
TENSORIZE_WORKERS = max(1, min(os.cpu_count() or 1, 16))
EVAL_LIMIT        = 500
TEST_LIMIT        = 200

print("=" * 72)
print("LEAN FLOW v2.7 CONFIG (HERMES + xLAM-60K 1-CALL)")
print("=" * 72)
print(f"EXPERIMENT_NAME   = {EXPERIMENT_NAME}  SEED = {SEED}")
print(f"BACKBONE          = {BACKBONE}")
print(f"MAX_SEQ_LEN       = {MAX_SEQ_LEN}  MAX_SPAN_WIDTH = {MAX_SPAN_WIDTH}  MAX_POS = {MAX_POS}")
print(f"DATA_FRACTION     = {DATA_FRACTION}  TENSORIZE_WORKERS = {TENSORIZE_WORKERS}")
print(f"XLAM_TRAIN_ROWS   = {XLAM_TRAIN_ROWS}  XLAM_DEV_ROWS = {XLAM_DEV_ROWS}  SINGLE_CALL_ONLY = {SINGLE_CALL_ONLY}")
print(f"EVAL_LIMIT        = {EVAL_LIMIT}  TEST_LIMIT = {TEST_LIMIT}")
print(f"DATA_DIR          = {DATA_DIR}")
print(f"TENSOR_DIR        = {TENSOR_DIR}")
print(f"RESULTS_DIR       = {RESULTS_DIR}")
print(f"DEVICE            = {DEVICE}")

LEAN FLOW v2.7 CONFIG (HERMES + xLAM-60K 1-CALL)
EXPERIMENT_NAME   = spancall_lean_modernbert  SEED = 42
BACKBONE          = answerdotai/ModernBERT-large
MAX_SEQ_LEN       = 4096  MAX_SPAN_WIDTH = 16  MAX_POS = 1
DATA_FRACTION     = 0.5  TENSORIZE_WORKERS = 12
XLAM_TRAIN_ROWS   = 15000  XLAM_DEV_ROWS = 1000  SINGLE_CALL_ONLY = True
EVAL_LIMIT        = 500  TEST_LIMIT = 200
DATA_DIR          = /content/spancall/data/processed_v2_7
TENSOR_DIR        = /content/spancall/data/tensorized_lean_v2_7
RESULTS_DIR       = /content/spancall/results_lean_v2_7
DEVICE            = cuda


In [3]:
# 0c. Runtime, GPU & Package Check (+ Optional Colab Drive Mount)
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo 'No NVIDIA GPU detected (using MPS / CPU fallback)'

import spancall
import spancall.lean as lean
import span_template
import transformers
import datasets

print("spancall     :", spancall.__file__)
print("spancall.lean:", lean.__file__)
print("span_template:", span_template.__file__)
print("transformers :", transformers.__version__, "| datasets:", datasets.__version__)

if DRIVE_MOUNT:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print("Drive mounted:", DRIVE_OUT)

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
spancall     : /content/spancall/spancall/__init__.py
spancall.lean: /content/spancall/spancall/lean/__init__.py
span_template: /content/spancall/span_template/__init__.py
transformers : 5.22.1 | datasets: 4.8.5


---
## 1. Dataset (`data/processed_v2_7/`) & Pre-Flight Gate

Downloads the raw splits from Hugging Face if missing, runs `scripts/realign_v2.py` to produce render-invariant `utt_rel_start`/`utt_rel_end` offsets, and verifies the template stamp (`Gate A`) and `utterance_span` offset validity (`Gate B`).

In [4]:
from huggingface_hub import snapshot_download
from span_template.stamp import check_stamp
from spancall.lean.diagnostics import audit_gt_offsets
import json, os, subprocess, sys

CANONICAL_SPLITS = [
    "hermes_train.jsonl",
    "hermes_dev.jsonl",
    "hermes_calib.jsonl",
    "hermes_eval.jsonl",
    "hermes_test_ood_schema.jsonl",
    "hermes_test_ood_para.jsonl",
    "hermes_test_adversarial.jsonl",
]

XLAM_SPLITS = [
    "xlam_train.jsonl",
    "xlam_dev.jsonl",
    "xlam_calib.jsonl",
    "xlam_eval.jsonl",
    "xlam_test_ood_schema.jsonl",
    "xlam_test_ood_para.jsonl",
    "xlam_test_adversarial.jsonl",
]

os.makedirs(DATA_DIR, exist_ok=True)
_missing_v2 = [f for f in CANONICAL_SPLITS + XLAM_SPLITS if not os.path.exists(os.path.join(DATA_DIR, f))]
if _missing_v2:
    # v2.7: never pull a prebuilt processed_v2_7. Object decomposition changes the
    # rendered param list, so a stale directory would train on v2.7-shaped labels
    # (one opaque `object` param, no dotted leaves) and every object number in
    # section 4d would be 0 for a reason that has nothing to do with the model.
    # Hermes splits are rebuilt by the realign block below; xLAM by prepare_data.
    print(f"[v2.7] {len(_missing_v2)} of {len(CANONICAL_SPLITS) + len(XLAM_SPLITS)} split(s) "
          f"missing in {DATA_DIR}")
    print("       no prebuilt v2 download is used in v2.7 - every split is rebuilt below.")

_already = all(os.path.exists(os.path.join(DATA_DIR, f)) for f in CANONICAL_SPLITS)
_stamp_ok = check_stamp(DATA_DIR).ok if _already else False
if FORCE_REALIGN or not _already or not _stamp_ok:
    os.makedirs(V1_DATA_DIR, exist_ok=True)
    missing_v1 = [f for f in CANONICAL_SPLITS if not os.path.exists(os.path.join(V1_DATA_DIR, f))]
    if missing_v1:
        snapshot_download(
            repo_id=HF_DATASET_ID,
            repo_type="dataset",
            local_dir=str(PROJECT_ROOT),
            allow_patterns=["data/processed/*.jsonl", "data/processed/*.json"],
            token=HF_TOKEN,
        )
    print(f"Re-aligning v1 -> {DATA_DIR} ...")
    _realign = [
        sys.executable, os.path.join(str(PROJECT_ROOT), "scripts", "realign_v2.py"),
        "--src_dir", V1_DATA_DIR,
        "--out_dir", DATA_DIR,
        "--gate_split", EVAL_SPLIT,
        "--max_utt_stale_rate", str(MAX_UTT_STALE_RATE),
        "--max_unresolvable", str(MAX_UNRESOLVABLE),
    ]
    if ALLOW_STALE_LABELS:
        _realign.append("--force")
    _rc = subprocess.run(_realign, cwd=str(PROJECT_ROOT)).returncode
    if _rc != 0:
        raise SystemExit(f"realign_v2 exited {_rc}. Inspect data/processed_v2_7/realign_report.json.")
else:
    print(f"v2 splits ready in {DATA_DIR}")

# xLAM-60k (1-call) splits: download public xlam_*.jsonl from convoicon/spancall
# and re-align through the v2.7 aligner (avoids the gated Salesforce/xlam-60k
# hub download unless the snapshot lacks them).
_missing_xlam = [f for f in XLAM_SPLITS if not os.path.exists(os.path.join(DATA_DIR, f))]
if _missing_xlam and (XLAM_TRAIN_ROWS is None or XLAM_TRAIN_ROWS > 0):
    _xlam_src_dir = os.path.join(str(PROJECT_ROOT), "data", "processed_v2")
    _missing_xlam_src = [f for f in XLAM_SPLITS if not os.path.exists(os.path.join(_xlam_src_dir, f))]
    if _missing_xlam_src:
        print(f"Downloading xLAM splits from {HF_DATASET_ID} ...")
        snapshot_download(
            repo_id=HF_DATASET_ID,
            repo_type="dataset",
            local_dir=str(PROJECT_ROOT),
            allow_patterns=["data/processed_v2" + "/xlam_*.jsonl", "data/processed_v2" + "/*.json"],
            token=HF_TOKEN,
        )
    if all(os.path.exists(os.path.join(_xlam_src_dir, f)) for f in XLAM_SPLITS):
        print(f"Re-aligning xLAM splits -> {DATA_DIR} ...")
        _xlam_realign = [
            sys.executable, os.path.join(str(PROJECT_ROOT), "scripts", "realign_v2.py"),
            "--src_dir", _xlam_src_dir,
            "--out_dir", DATA_DIR,
            "--splits", *[os.path.splitext(f)[0] for f in XLAM_SPLITS],
            "--gate_split", "xlam_eval",
            "--max_utt_stale_rate", str(MAX_UTT_STALE_RATE),
            "--max_unresolvable", str(MAX_UNRESOLVABLE),
        ]
        if ALLOW_STALE_LABELS:
            _xlam_realign.append("--force")
        _xrc = subprocess.run(_xlam_realign, cwd=str(PROJECT_ROOT)).returncode
        if _xrc != 0:
            raise SystemExit(f"realign_v2 (xlam) exited {_xrc}. Inspect data/processed_v2_7/realign_report.json.")
    else:
        print(f"Preparing xLAM-60k (1-call) splits into {DATA_DIR} via scripts/prepare_data.py ...")
        _prep_rc = subprocess.run([
            sys.executable, os.path.join(str(PROJECT_ROOT), "scripts", "prepare_data.py"),
            "--sources", "xlam",
            "--output_dir", DATA_DIR,
        ], cwd=str(PROJECT_ROOT)).returncode
        if _prep_rc != 0:
            raise SystemExit(f"prepare_data.py --sources xlam exited {_prep_rc} (set HF_TOKEN if Salesforce/xlam-function-calling-60k is gated)")
        # v2.7 provenance note: template_stamp.json is directory-level, so the
        # stamp now attests the xLAM splits prepare_data just wrote, not the
        # Hermes splits realigned above. Gate A below still validates the
        # template REVISION (the thing that invalidated v1), nothing more.
        print("[Gate A / note] prepare_data.py re-stamped " + DATA_DIR + " -> Hermes splits are no longer attested by that stamp.")

# Pre-Flight Gate A: template stamp
_verdict = check_stamp(DATA_DIR)
print(f"[Gate A / Stamp] {_verdict.status}: {_verdict.message()}")
if not _verdict.ok and not ALLOW_STALE_LABELS:
    raise SystemExit("PRE-FLIGHT FAILED (template stamp): " + _verdict.message())

# Pre-Flight Gate B: offset audit on every split this notebook later
# reports on, not just eval+dev. The OOD / adversarial splits come from
# the same aligner, so an offset defect there has to fail here rather
# than surface as a mysteriously low section-5 score.
_audit_splits = [EVAL_SPLIT, "hermes_dev"]
for _osp in list(TEST_SPLITS) + ["xlam_dev"]:
    if _osp in _audit_splits:
        continue
    if os.path.exists(os.path.join(DATA_DIR, f"{_osp}.jsonl")):
        _audit_splits.append(_osp)
for _split in _audit_splits:
    _m = audit_gt_offsets(_split, DATA_DIR)
    _stale = _m.get("utterance_span_stale_rate")
    _unres = _m.get("unresolvable_rate")
    _n_utt = int(_m.get("utterance_span_targets") or 0)
    if _n_utt == 0 and "adversarial" in _split:
        _ok = True  # marker-injection splits have target_spans={} by contract
    else:
        _ok = (
            isinstance(_stale, float) and _stale == _stale and _stale < MAX_UTT_STALE_RATE
            and ((_unres is None) or (isinstance(_unres, float) and _unres == _unres and _unres < MAX_UNRESOLVABLE))
        )
    print(f"[Gate B / Audit] {_split:<16} utterance_span_stale_rate={_stale} "
          f"({_m.get('utterance_span_targets')} targets), unresolvable_rate={_unres} -> {'PASS' if _ok else 'FAIL'}")
    if not _ok and not ALLOW_STALE_LABELS:
        raise SystemExit(f"PRE-FLIGHT FAILED on {_split}")

# Pre-Flight Gate C (v2.7, warning-only): is object decomposition present?
# If the labels are still v2.6-shaped (one opaque `object` param, no dotted
# leaves) then every 'object-leaf' figure in section 4d is structurally 0
# and the v2.7 question cannot be answered by this run. Warn loudly instead
# of aborting, because the label-side work is plan steps 2-3.
from spancall.data.schema import UnifiedSample as _C_Sample
from spancall.data.flatten import PATH_SEP as _C_SEP
_c_roots = _c_leaves = _c_opaque = 0
_c_path = os.path.join(DATA_DIR, f"{EVAL_SPLIT}.jsonl")
if os.path.exists(_c_path):
    with open(_c_path) as _fh_c:
        for _i_c, _line_c in enumerate(_fh_c):
            if _i_c >= 300:
                break
            if not _line_c.strip():
                continue
            _s_c = _C_Sample.from_dict(json.loads(_line_c))
            for _t_c in _s_c.tools:
                _c_roots += len(_t_c.object_paths or {})
                for _p_c in _t_c.parameters:
                    if _C_SEP in _p_c.name:
                        _c_leaves += 1
                    elif (_p_c.param_type or "").lower() == "object":
                        _c_opaque += 1
if _c_roots + _c_leaves + _c_opaque == 0:
    print("[Gate C / Decompose] no object-bearing tool in "
          + EVAL_SPLIT + " - nothing to check.")
else:
    _c_ok = _c_roots > 0 and _c_opaque == 0
    print(f"[Gate C / Decompose] roots={_c_roots} leaves={_c_leaves} opaque={_c_opaque} -> {'PASS' if _c_ok else 'WARN'}")
    if not _c_ok:
        print("  [!] labels look v2.6-shaped for object params: opaque `object`")
        print("      params outnumber dotted leaves. Section 4d will report zero")
        print("      object-leaf slots until plan steps 2-3 land (nested schema")
        print("      parsing + aligner object case). Do not read 4d as model quality.")

v2 splits ready in /content/spancall/data/processed_v2_7
[Gate A / Stamp] ok: template stamp OK (1e62a355a351)
[Gate B / Audit] hermes_eval      utterance_span_stale_rate=0.0 (3771 targets), unresolvable_rate=0.0 -> PASS
[Gate B / Audit] hermes_dev       utterance_span_stale_rate=0.0 (187 targets), unresolvable_rate=0.0 -> PASS
[Gate B / Audit] hermes_test_ood_schema utterance_span_stale_rate=0.0 (2605 targets), unresolvable_rate=0.0 -> PASS
[Gate B / Audit] hermes_test_ood_para utterance_span_stale_rate=0.0 (2562 targets), unresolvable_rate=0.0 -> PASS
[Gate B / Audit] hermes_test_adversarial utterance_span_stale_rate=nan (0 targets), unresolvable_rate=nan -> PASS
[Gate B / Audit] xlam_eval        utterance_span_stale_rate=0.0 (17005 targets), unresolvable_rate=0.0 -> PASS
[Gate B / Audit] xlam_test_ood_schema utterance_span_stale_rate=0.0 (8927 targets), unresolvable_rate=0.0 -> PASS
[Gate B / Audit] xlam_dev         utterance_span_stale_rate=0.0 (4066 targets), unresolvable_rate=0.0

---
## 2. Lean Tensorizer — Hermes + xLAM-60k (1-Call) Mix

Tensorizes `hermes_train.jsonl` + `xlam_train.jsonl` (up to `XLAM_TRAIN_ROWS`) into `t2.jsonl` and `hermes_dev.jsonl` + `xlam_dev.jsonl` (up to `XLAM_DEV_ROWS`) into `dev.jsonl` using a shared multiprocessing fork pool.

In [5]:
from typing import Any, Dict, List, Optional
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor
import json, math, multiprocessing as mp, os, queue, random, time
import yaml
from transformers import AutoTokenizer

from span_template import SPECIAL_MARKER_TOKENS
from spancall.data.schema import UnifiedSample
from spancall.model.masks import CandidateMaskBuilder
from spancall.lean.tensorize import ToolPool, tensorize_sample_lean, has_positive_targets

tokenizer = AutoTokenizer.from_pretrained(BACKBONE, use_fast=True)
tokenizer.add_special_tokens({"additional_special_tokens": list(SPECIAL_MARKER_TOKENS)})
mask_builder = CandidateMaskBuilder(max_seq_len=MAX_SEQ_LEN, max_span_width=MAX_SPAN_WIDTH)

PACK_TOOLS = 3
try:
    with open(os.path.join(str(PROJECT_ROOT), "configs", "default.yaml")) as _fh:
        _default_cfg = yaml.safe_load(_fh) or {}
    PACK_TOOLS = int((_default_cfg.get("data") or {}).get("pack_tools", 3) or 3)
except Exception as _exc:
    print(f"[!] configs/default.yaml unreadable ({_exc}); using PACK_TOOLS={PACK_TOOLS}")

TOOL_POOL = None
PACK_TOOLS_EPH = 1

def _iter_raw_lines(path: str):
    with open(path, "r") as fh:
        for line in fh:
            line = line.strip()
            if line:
                yield line

def _iter_raw_dicts(path: str):
    for line in _iter_raw_lines(path):
        yield json.loads(line)

def _fraction_subset(raw_lines, frac: float, seed: int):
    raw_lines = list(raw_lines)
    if frac >= 1.0:
        return raw_lines
    rng = random.Random(int(seed))
    n = max(int(len(raw_lines) * frac), 1)
    keep = set(rng.sample(range(len(raw_lines)), min(n, len(raw_lines))))
    return [d for i, d in enumerate(raw_lines) if i in keep]

def _serialize(row):
    return {k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in row.items()}

def _tensorize_lean_task(args):
    raw_line, pack_multi = args
    drops: Dict[str, int] = {}
    errors: Dict[str, int] = {}
    try:
        d = json.loads(raw_line)
        s = UnifiedSample.from_dict(d)
        ts_stats: Dict[str, Any] = {}
        row = tensorize_sample_lean(
            s,
            tokenizer=tokenizer,
            mask_builder=mask_builder,
            max_seq_len=MAX_SEQ_LEN,
            max_cands=MAX_CANDS,
            max_pos=int(MAX_POS),
            pack_multi_tools=pack_multi,
            stats=ts_stats,
            tool_pool=TOOL_POOL,
            pack_tools=PACK_TOOLS_EPH,
            max_tools=MAX_TOOLS,
            jitter=JITTER_PACK_TOOLS,
            include_abstain=True,
        )
        drops = dict(ts_stats.get("param_drop_reasons", {}))
        for _t, _cnt in (ts_stats.get("render_T_hist") or {}).items():
            _k = f"render_T={int(_t)}"
            drops[_k] = drops.get(_k, 0) + int(_cnt)
        if row is None or (not has_positive_targets(row) and not bool(row.get("is_abstain", 0))):
            return None, drops, errors
        return json.dumps(_serialize(row)), drops, errors
    except Exception as exc:
        cls = type(exc).__name__
        errors[cls] = errors.get(cls, 0) + 1
        return None, drops, errors

_WINDOW = 2048

def _drain_ordered(pool, raw, pack_multi: bool, absorb, done, rows_len, need,
                   workers: int = 1, window: int = _WINDOW):
    n = len(raw)
    guard = max(2 * int(workers), 16)
    finished_q: queue.SimpleQueue = queue.SimpleQueue()
    buffer: Dict[int, Any] = {}
    nxt = 0
    submit = 0
    inflight = 0
    while nxt < n and not done():
        budget = max(0, need - rows_len())
        cap = min(window, budget + guard)
        while submit < n and (submit - nxt) < cap and not done():
            fut = pool.submit(_tensorize_lean_task, (raw[submit], pack_multi))
            fut.add_done_callback(lambda f, i=submit: finished_q.put((i, f)))
            submit += 1
            inflight += 1
        if inflight == 0:
            break
        idx, fut = finished_q.get()
        inflight -= 1
        buffer[idx] = fut.result()
        while nxt in buffer and not done():
            absorb(buffer.pop(nxt))
            nxt += 1

def _tensorize_many(raw, pack_multi: bool, need, workers: int, executor=None):
    if need is None:
        need = len(raw)
    rows: List[str] = []
    drops_total: Counter = Counter()
    errors_total: Counter = Counter()

    def _absorb(res):
        line, drops, errors = res
        for k, v in (drops or {}).items():
            drops_total[k] += v
        for k, v in (errors or {}).items():
            errors_total[k] += v
        if line is not None and len(rows) < need:
            rows.append(line)

    def _done():
        return len(rows) >= need

    if executor is not None:
        _drain_ordered(executor, raw, pack_multi, _absorb, _done,
                       rows_len=lambda: len(rows), need=need, workers=workers)
        return rows, drops_total, errors_total

    for line in raw:
        _absorb(_tensorize_lean_task((line, pack_multi)))
        if _done():
            break
    return rows, drops_total, errors_total

def _filter_single_call_lines(raw_lines):
    if not SINGLE_CALL_ONLY:
        return list(raw_lines), 0
    kept = []
    dropped = 0
    for line in raw_lines:
        d = json.loads(line)
        if len(d.get("ground_truth_calls") or []) == 1:
            kept.append(line)
        else:
            dropped += 1
    return kept, dropped

def _cap_lines(raw_lines, cap: Optional[int], seed: int):
    raw_lines = list(raw_lines)
    if cap is None or cap >= len(raw_lines):
        return raw_lines
    if cap <= 0:
        return []
    rng = random.Random(int(seed))
    keep = set(rng.sample(range(len(raw_lines)), cap))
    return [d for i, d in enumerate(raw_lines) if i in keep]

os.makedirs(TENSOR_DIR, exist_ok=True)
_t2_sources = SPLIT_INPUTS.get("t2") or next(iter(SPLIT_INPUTS.values()))
if isinstance(_t2_sources, str):
    _t2_sources = [_t2_sources]
if PACK_TOOLS > 1:
    _pool_dicts = []
    for _src_file in _t2_sources:
        _src_path = os.path.join(DATA_DIR, _src_file)
        if not os.path.exists(_src_path):
            continue
        if _src_file.startswith("xlam_"):
            if XLAM_TRAIN_ROWS is not None and XLAM_TRAIN_ROWS <= 0:
                continue
            for _l in _cap_lines(_iter_raw_lines(_src_path), XLAM_TRAIN_ROWS, SEED):
                _pool_dicts.append(json.loads(_l))
        else:
            _pool_dicts.extend(_iter_raw_dicts(_src_path))
    if _pool_dicts:
        TOOL_POOL = ToolPool.from_row_dicts(_pool_dicts, seed=SEED)
        print(f"[tool-pool] built from {_t2_sources}: {len(TOOL_POOL)} schemas "
              f"(collisions={len(TOOL_POOL.name_collisions)}, pack_tools={PACK_TOOLS})")
PACK_TOOLS_EPH = int(PACK_TOOLS) if TOOL_POOL is not None else 1

_fork_ctx = None
_shared_ex = None
if TENSORIZE_WORKERS > 1:
    try:
        _fork_ctx = mp.get_context("fork")
        _shared_ex = ProcessPoolExecutor(max_workers=TENSORIZE_WORKERS, mp_context=_fork_ctx)
    except Exception as _exc:
        print(f"[!] fork pool unavailable ({_exc}); running single-process")

try:
    for out_name, in_spec in SPLIT_INPUTS.items():
        in_files = [in_spec] if isinstance(in_spec, str) else list(in_spec)
        raw: List[str] = []
        per_src_counts: Dict[str, int] = {}
        for in_name in in_files:
            src = os.path.join(DATA_DIR, in_name)
            if not os.path.exists(src):
                per_src_counts[in_name] = 0
                continue
            filtered_lines, n_multi_drop = _filter_single_call_lines(_iter_raw_lines(src))
            if in_name.startswith("xlam_"):
                cap = XLAM_TRAIN_ROWS if out_name == "t2" else XLAM_DEV_ROWS
                part = _cap_lines(filtered_lines, cap, SEED)
            else:
                part = _fraction_subset(filtered_lines, DATA_FRACTION, SEED)
            per_src_counts[in_name] = len(part)
            if n_multi_drop:
                print(f"  [{out_name}/{in_name}] SINGLE_CALL_ONLY dropped {n_multi_drop} multi-call rows")
            raw.extend(part)
        if len(in_files) > 1:
            _shuf_rng = random.Random(int(SEED))
            _shuf_rng.shuffle(raw)
        limit = SAMPLE_LIMITS.get(out_name)
        t0 = time.perf_counter()
        rows, drops, errors = _tensorize_many(raw, True, limit, TENSORIZE_WORKERS, executor=_shared_ex)
        dt = time.perf_counter() - t0
        out_path = os.path.join(TENSOR_DIR, f"{out_name}.jsonl")
        with open(out_path, "w") as fh:
            fh.write("\n".join(rows) + ("\n" if rows else ""))
        render_hist = {k.split("=")[1]: v for k, v in drops.items() if k.startswith("render_T=")}
        print(f"[tensorize] {out_name:<4} <- {per_src_counts}: {len(rows):>6} rows in {dt:.1f}s "
              f"({len(rows)/max(dt, 1e-6):.1f} rows/s) | render_T={render_hist} "
              f"| match_missed={drops.get('match_missed', 0)} | errors={dict(errors)}")
finally:
    if _shared_ex is not None:
        _shared_ex.shutdown(wait=True)

fingerprint = {
    "experiment_name": EXPERIMENT_NAME,
    "seed": SEED,
    "backbone": BACKBONE,
    "max_seq_len": MAX_SEQ_LEN,
    "max_span_width": MAX_SPAN_WIDTH,
    "max_pos": MAX_POS,
    "data_fraction": DATA_FRACTION,
    "xlam_train_rows": XLAM_TRAIN_ROWS,
    "xlam_dev_rows": XLAM_DEV_ROWS,
    "split_inputs": SPLIT_INPUTS,
    "sample_limits": SAMPLE_LIMITS,
    "lean_flow": True,
}
fp_path = os.path.join(TENSOR_DIR, "config_fingerprint.json")
with open(fp_path, "w") as fh:
    json.dump(fingerprint, fh, indent=2, sort_keys=True)
print("wrote config fingerprint ->", fp_path)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

[tool-pool] built from ['hermes_train.jsonl', 'xlam_train.jsonl']: 4854 schemas (collisions=2118, pack_tools=3)
  [t2/hermes_train.jsonl] SINGLE_CALL_ONLY dropped 3966 multi-call rows
[tensorize] t2   <- {'hermes_train.jsonl': 10052, 'xlam_train.jsonl': 15000}:  20779 rows in 38.4s (541.5 rows/s) | render_T={'4': 2239, '2': 4470, '1': 7870, '3': 7154, '7': 321, '6': 401, '8': 341, '5': 722} | match_missed=2504 | errors={}
  [dev/hermes_dev.jsonl] SINGLE_CALL_ONLY dropped 32 multi-call rows
[tensorize] dev  <- {'hermes_dev.jsonl': 844, 'xlam_dev.jsonl': 1000}:    850 rows in 1.2s (683.5 rows/s) | render_T={'1': 406, '2': 278, '4': 166, '3': 429, '8': 9, '5': 40, '7': 6, '6': 9} | match_missed=53 | errors={}
wrote config fingerprint -> /content/spancall/data/tensorized_lean_v2_7/config_fingerprint.json


---
## 3. Training — Lean Flow (`scripts/train_lean.py`)

Trains `LeanSpanCallModel` (`ModernBERT-large` + `ToolSelectionHead` H1 + `LeanPointerHead` H3) on stage `T2` with full data (`TRAIN_FRACTION = 1.0`), effective batch size 128, `bf16`, and best-checkpoint restoration.

In [6]:
import os, yaml
from scripts.train_lean import main as train_lean_main

TRAIN_FRACTION = 1.0
NUM_EPOCHS = 3
MICRO_BATCH_SIZE = 32
EFFECTIVE_BATCH_SIZE = 128

with open("configs/default.yaml") as fh:
    cfg = yaml.safe_load(fh)

cfg["experiment"]["name"] = EXPERIMENT_NAME
cfg["experiment"]["seed"] = SEED
cfg["experiment"]["output_dir"] = RESULTS_DIR
cfg["experiment"]["data_dir"] = TENSOR_DIR

cfg["optimization"]["num_epochs"] = NUM_EPOCHS
cfg["optimization"]["batch_size"] = MICRO_BATCH_SIZE
cfg["optimization"]["effective_batch_size"] = EFFECTIVE_BATCH_SIZE
cfg["optimization"]["max_tokens_per_batch"] = 4096
cfg["optimization"]["optimizer"] = "adamw"
cfg["optimization"]["scheduler"] = "cosine_warmup"
cfg["optimization"]["min_lr_ratio"] = 0.10
cfg["optimization"]["early_stopping_patience"] = 1
cfg["optimization"]["restore_best_on_exit"] = True
cfg["optimization"]["mixed_precision"] = "bf16"

cfg["training"]["gradient_accumulation_steps"] = max(int(round(EFFECTIVE_BATCH_SIZE / MICRO_BATCH_SIZE)), 1)
cfg["training"]["gradient_checkpointing"] = False
cfg["training"]["eval_splits"] = ["dev"]
cfg["training"]["eval_limit"] = 2000
cfg["training"]["freeze_encoder_stages"] = []

cfg["model"]["backbone_name"] = BACKBONE
cfg["model"]["max_seq_len"] = MAX_SEQ_LEN

_EFFECTIVE_PACK_TOOLS = int(globals().get("PACK_TOOLS_EPH", PACK_TOOLS) or 1)
cfg.setdefault("data", {})["fraction"] = TRAIN_FRACTION
cfg["data"]["pack_tools"] = _EFFECTIVE_PACK_TOOLS

CONFIG_PATH = "configs/colab_lean_v2_7.yaml"
with open(CONFIG_PATH, "w") as fh:
    yaml.safe_dump(cfg, fh)
print(f"wrote {CONFIG_PATH} (backbone={BACKBONE}, epochs={NUM_EPOCHS}, "
      f"micro_bs={MICRO_BATCH_SIZE}, effective_bs={EFFECTIVE_BATCH_SIZE}, "
      f"data_fraction={TRAIN_FRACTION}, pack_tools={_EFFECTIVE_PACK_TOOLS})")

train_lean_main([
    "--config", CONFIG_PATH,
    "--experiment_name", EXPERIMENT_NAME,
    "--output_dir", RESULTS_DIR,
    "--data_dir", TENSOR_DIR,
    "--stages", "T2",
    "--device", DEVICE,
    "--seed", str(SEED),
])

wrote configs/colab_lean_v2_7.yaml (backbone=answerdotai/ModernBERT-large, epochs=3, micro_bs=32, effective_bs=128, data_fraction=1.0, pack_tools=3)
[SpanCall Lean Train] config: configs/colab_lean_v2_7.yaml
[SpanCall Lean Train] backbone=answerdotai/ModernBERT-large


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.58GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


[lean-train:T2] start: rows=20779 epochs=3 steps/epoch=2966 total_steps=8898 bs=32 accum=4 effective_bs=128 opt_steps=2225 lr_floor_ratio=0.10


lean-train[T2]:   0%|          | 0/8898 [00:00<?, ?it/s]

[lean-train:T2] epoch 1/3 done: mean_loss=1.8925 (batchmean=1.9624) ptr=1.7217 tool=0.1708
[lean-eval:dev] span_acc=0.7404 (n=1564) loss=0.8411 (batchmean=0.8205) ptr=0.8245 tool=0.0166 tool_acc=0.9965 (n=850) candidates/row=2.43
[lean-train:T2:epoch 1] NEW BEST on dev: exact_call=0.6753 -> saved to /content/spancall/results_lean_v2_7/spancall_lean_modernbert_seed42/best_checkpoint
[lean-train:T2] epoch 2/3 done: mean_loss=0.6076 (batchmean=0.6143) ptr=0.6031 tool=0.0045
[lean-eval:dev] span_acc=0.8146 (n=1564) loss=0.6402 (batchmean=0.6390) ptr=0.6282 tool=0.0120 tool_acc=0.9965 (n=850) candidates/row=2.43
[lean-train:T2:epoch 2] NEW BEST on dev: exact_call=0.7588 -> saved to /content/spancall/results_lean_v2_7/spancall_lean_modernbert_seed42/best_checkpoint
[lean-train:T2] epoch 3/3 done: mean_loss=0.4212 (batchmean=0.4227) ptr=0.4204 tool=0.0008
[lean-eval:dev] span_acc=0.8306 (n=1564) loss=0.6126 (batchmean=0.6035) ptr=0.6032 tool=0.0095 tool_acc=0.9953 (n=850) candidates/row=2.43


0

In [7]:
# 3b. Training Telemetry, Best Checkpoint Selection & LeanResolver Init
import glob, json, math, os
from typing import Any, Dict, List, Optional
import torch
from transformers import AutoTokenizer

from span_template import SPECIAL_MARKER_TOKENS
from spancall.lean.model import LeanSpanCallModel
from spancall.lean.resolver import LeanResolver
from spancall.model.encoder import SpanCallTransformerEncoder
from spancall.train.config import TrainingConfig

def _num(rec, key):
    if rec is None:
        return None
    v = rec.get(key)
    if v is None:
        return None
    try:
        f = float(v)
    except (TypeError, ValueError):
        return None
    return None if math.isnan(f) else f

def _fmt_cell(v, width, fmt=".3f"):
    return f"{'—':>{width}}" if v is None else f"{v:>{width}{fmt}}"

exp_root = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_seed{SEED}")
log_path = os.path.join(exp_root, "train_log.jsonl")
if os.path.exists(log_path):
    with open(log_path) as fh:
        train_log = [json.loads(l) for l in fh if l.strip()]
    evals = [r for r in train_log if r.get("event") == "epoch_eval"]
    ends  = [r for r in train_log if r.get("event") == "epoch_end"]
    print("=" * 84)
    print("PER-EPOCH HELD-OUT DEV CURVE")
    print("=" * 84)
    print(f"  {'ep':>3}{'n':>6}{'span_acc':>10}{'params_ok':>11}{'exact_call':>12}"
          f"{'exact_T1':>10}{'exact_T2+':>10}{'tool_all':>10}{'tool_T2+':>10}{'train_loss':>12}{'dev_loss':>10}")
    for r in evals:
        ep = r.get("epoch", "?")
        end_r = next((e for e in ends if e.get("epoch") == ep), None)
        print(f"  {ep:>3}{int(_num(r, 'n_span') or _num(r, 'n') or 0):>6}"
              f"{_fmt_cell(_num(r, 'span_acc'), 10)}"
              f"{_fmt_cell(_num(r, 'exact_params_only'), 11)}"
              f"{_fmt_cell(_num(r, 'exact_call'), 12)}"
              f"{_fmt_cell(_num(r, 'exact_call_t1'), 10)}"
              f"{_fmt_cell(_num(r, 'exact_call_multi'), 10)}"
              f"{_fmt_cell(_num(r, 'tool_acc'), 10)}"
              f"{_fmt_cell(_num(r, 'tool_acc_multi'), 10)}"
              f"{_fmt_cell(_num(end_r, 'mean_loss'), 12, '.4f')}"
              f"{_fmt_cell(_num(r, 'eval_loss'), 10, '.4f')}")

best_cp = os.path.join(exp_root, "best_checkpoint")
if os.path.exists(os.path.join(best_cp, "model.pt")):
    CHECKPOINT_DIR = best_cp
else:
    cps = sorted(glob.glob(os.path.join(exp_root, "**", "model.pt"), recursive=True))
    CHECKPOINT_DIR = os.path.dirname(cps[-1]) if cps else None
if CHECKPOINT_DIR is None:
    # LeanResolver(model_path=None) skips load_checkpoint entirely, so the
    # eval below would score randomly initialised weights and still print
    # plausible-looking numbers. Refuse instead.
    raise SystemExit(
        "no model.pt under " + exp_root + ": training persisted no checkpoint. "
        "LeanResolver(model_path=None) would run UNTRAINED weights, so every "
        "metric in sections 3-5 would be noise."
    )
print("\nCHECKPOINT_DIR =", CHECKPOINT_DIR)

cfg = TrainingConfig.from_yaml(CONFIG_PATH)
tokenizer = AutoTokenizer.from_pretrained(cfg.model.backbone_name, use_fast=True)
encoder = SpanCallTransformerEncoder(
    model_name_or_path=cfg.model.backbone_name,
    hidden_dropout_prob=cfg.model.dropout,
    tokenizer=tokenizer,
)
encoder.add_marker_tokens(list(SPECIAL_MARKER_TOKENS))
model = LeanSpanCallModel(
    encoder=encoder,
    head_dim=cfg.model.head_dim,
    max_span_width=cfg.model.max_span_width,
    hidden_dim=encoder.hidden_dim,
)

MAX_CALLS = 1 if SINGLE_CALL_ONLY else int(getattr(cfg.data, "max_calls", 1))
CALL_MIN_PROB = float(getattr(cfg.data, "call_min_prob", 0.25))
CALL_MIN_PROB_RATIO = float(getattr(cfg.data, "call_min_prob_ratio", 0.5))

resolver = LeanResolver(
    model_path=CHECKPOINT_DIR,
    encoder=encoder,
    model=model,
    tokenizer=tokenizer,
    max_seq_len=cfg.model.max_seq_len,
    max_span_width=cfg.model.max_span_width,
    max_calls=MAX_CALLS,
    call_min_prob=CALL_MIN_PROB,
    call_min_prob_ratio=CALL_MIN_PROB_RATIO,
    min_joint_margin=MIN_JOINT_MARGIN,
    collapse_guard=True,
    prevent_param_collapse=True,
)
if DEVICE in ("cuda", "mps"):
    model.to(DEVICE)
model.eval()
if hasattr(encoder, "eval"):
    encoder.eval()
print("LeanResolver ready.")

PER-EPOCH HELD-OUT DEV CURVE
   ep     n  span_acc  params_ok  exact_call  exact_T1 exact_T2+  tool_all  tool_T2+  train_loss  dev_loss
    1  1564     0.740      0.676       0.675     0.645     0.688     0.996     0.995      1.8925    0.8411
    2  1564     0.815      0.760       0.759     0.734     0.769     0.996     0.995      0.6076    0.6402
    3  1564     0.831      0.782       0.780     0.738     0.797     0.995     0.993      0.4212    0.6126

CHECKPOINT_DIR = /content/spancall/results_lean_v2_7/spancall_lean_modernbert_seed42/best_checkpoint


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LeanResolver ready.


---
## 4. Step-by-Step Evaluation (`hermes_eval`)

### 4.0 Stratified Benchmark Slice & Step 0 (Pre-Encode Sanitize Gate)

Builds the shared deterministic stratified slice (`subset_hermes_eval.jsonl` across `F_func`, `F_json`, `A_clean`, `A_adv`) and runs the Step 0 pre-encode marker-injection check.

In [8]:
import json, os
from span_template.template import RecentRow, build_input_multi
from span_template.sanitize import SanitizeError
from spancall.data.schema import UnifiedSample
from spancall.data.align import tooldef_to_template_tool
from spancall.lean.eval_p1 import build_stratified_eval_subset, classify_eval_cohort

SUBSET_EVAL_PATH = os.path.join(RESULTS_DIR, f"subset_{EVAL_SPLIT}.jsonl")
CLEAN_EVAL_PATH  = os.path.join(RESULTS_DIR, f"subset_{EVAL_SPLIT}_clean.jsonl")
_slice_info = build_stratified_eval_subset(
    EVAL_SPLIT,
    data_dir=DATA_DIR,
    out_path=SUBSET_EVAL_PATH,
    limit=EVAL_LIMIT,
    seed=SEED,
)
if SINGLE_CALL_ONLY:
    with open(SUBSET_EVAL_PATH, encoding="utf-8") as _fh:
        _one_call_lines = [
            _l for _l in _fh
            if _l.strip() and len((json.loads(_l).get("ground_truth_calls") or [])) == 1
        ]
    with open(SUBSET_EVAL_PATH, "w", encoding="utf-8") as _fh:
        _fh.writelines(_one_call_lines)
    print(f"[SINGLE_CALL_ONLY] filtered {EVAL_SPLIT} slice -> {len(_one_call_lines)} 1-call rows")
print("=== 4.0 Stratified Benchmark Slice ===")
print("Available by cohort:", _slice_info["available_by_cohort"])
print("Sampled   by cohort:", _slice_info["sampled_by_cohort"], f"(total={_slice_info['sampled_total']})")

_step0_stats = {c: {"n": 0, "vetoed": 0} for c in ("F_func", "F_json", "A_clean", "A_adv")}
_clean_lines = []
with open(SUBSET_EVAL_PATH, encoding="utf-8") as _fh:
    for _line in _fh:
        _raw = _line.strip()
        if not _raw:
            continue
        _d = json.loads(_raw)
        _cohort = classify_eval_cohort(_d)
        _s = UnifiedSample.from_dict(_d)
        _st = _step0_stats.setdefault(_cohort, {"n": 0, "vetoed": 0})
        _st["n"] += 1
        _vetoed = False
        try:
            _recent = [
                RecentRow(key=str(r.get("key", "")), value=str(r.get("value", "")), source=str(r.get("source", "recent")))
                for r in (_s.recent_context or [])
            ]
            build_input_multi([tooldef_to_template_tool(t) for t in (_s.tools or [])], _s.utterance, recent_context=_recent)
        except SanitizeError:
            _vetoed = True
        except Exception:
            _vetoed = False
        if _vetoed:
            _st["vetoed"] += 1
        else:
            _clean_lines.append(_raw)

with open(CLEAN_EVAL_PATH, "w", encoding="utf-8") as _fh:
    for _l in _clean_lines:
        _fh.write(_l + "\n")

print("\n=== Step 0: Pre-Encode Sanitize Gate ===")
for _c in ("F_func", "F_json", "A_clean", "A_adv"):
    _st = _step0_stats.get(_c, {"n": 0, "vetoed": 0})
    _rate = (_st["vetoed"] / _st["n"]) if _st["n"] else 0.0
    print(f"  {_c:<8}: vetoed={_st['vetoed']:>3} / {_st['n']:<3} ({_rate:.1%})")
print(f"Non-vetoed clean eval rows -> {CLEAN_EVAL_PATH} ({len(_clean_lines)} rows)")

[SINGLE_CALL_ONLY] filtered hermes_eval slice -> 415 1-call rows
=== 4.0 Stratified Benchmark Slice ===
Available by cohort: {'F_func': 1485, 'F_json': 64, 'A_clean': 445, 'A_adv': 788, 'A_missing_gt_schema': 644}
Sampled   by cohort: {'F_func': 250, 'F_json': 50, 'A_clean': 150, 'A_adv': 50} (total=500)

=== Step 0: Pre-Encode Sanitize Gate ===
  F_func  : vetoed=  0 / 209 (0.0%)
  F_json  : vetoed=  0 / 50  (0.0%)
  A_clean : vetoed=  0 / 111 (0.0%)
  A_adv   : vetoed=  0 / 45  (0.0%)
Non-vetoed clean eval rows -> /content/spancall/results_lean_v2_7/subset_hermes_eval_clean.jsonl (415 rows)


---
### 4a. Step 1 in Isolation — Tool Selection (`H1`)

Scores `resolver.predict_tool` on native multi-tool rows (`len(s.tools) >= 2`), single-tool rows augmented with random pool distractors (`T=3`), and measures `F_func` vs `A_clean` score/margin separation.

In [9]:
import json, os, random, zlib
from collections import Counter, defaultdict
from spancall.data.schema import UnifiedSample
from spancall.lean.eval_p1 import classify_eval_cohort

def _pct(vals, q):
    if not vals:
        return float("nan")
    s = sorted(vals)
    idx = min(len(s) - 1, max(0, int(round((len(s) - 1) * q))))
    return s[idx]

def eval_tool_prediction_step1(subset_path: str, pool_path: str, pack_k: int = 3):
    with open(subset_path, encoding="utf-8") as fh:
        raw_dicts = [json.loads(l) for l in fh if l.strip()]

    pool = {}
    if os.path.exists(pool_path):
        with open(pool_path, encoding="utf-8") as fh:
            for l in fh:
                if not l.strip():
                    continue
                for t in (json.loads(l).get("tools") or []):
                    if t.get("name") and t["name"] not in pool:
                        pool[t["name"]] = UnifiedSample.from_dict(
                            {"sample_id": "p", "utterance": "", "tools": [t], "ground_truth_calls": []}
                        ).tools[0]
    pool_names = sorted(pool)

    native_stats = {"F_func": [0, 0], "A_clean": [0, 0], "other": [0, 0]}
    by_t = defaultdict(lambda: [0, 0])
    aug_stats = [0, 0]
    probs_by_cohort = defaultdict(list)
    margins_by_cohort = defaultdict(list)
    wrong_pairs = Counter()

    for d in raw_dicts:
        cohort = classify_eval_cohort(d)
        s = UnifiedSample.from_dict(d)
        if not s.ground_truth_calls or not s.tools:
            continue
        gt_name = s.ground_truth_calls[0].tool_name
        native_multi = len(s.tools) >= 2
        candidates = list(s.tools)
        if not native_multi and pool_names:
            seen = {t.name for t in candidates}
            salt = zlib.crc32(f"step1::{SEED}::{s.sample_id}".encode("utf-8"))
            rng = random.Random(salt)
            for pick in rng.sample(pool_names, min(len(pool_names), pack_k + 3)):
                if len(candidates) >= pack_k:
                    break
                if pick not in seen:
                    candidates.append(pool[pick])
                    seen.add(pick)
            rng.shuffle(candidates)
        if len(candidates) < 2:
            continue

        pred = resolver.predict_tool(candidates, s.utterance, s.recent_context, s.current_time_iso)
        pred_name = pred.get("tool_name")
        ok = bool(pred_name) and pred_name == gt_name

        bucket = cohort if cohort in native_stats else "other"
        if native_multi:
            native_stats[bucket][1] += 1
            native_stats[bucket][0] += int(ok)
            t_key = str(len(candidates)) if len(candidates) <= 4 else "5+"
            by_t[t_key][1] += 1
            by_t[t_key][0] += int(ok)
        else:
            aug_stats[1] += 1
            aug_stats[0] += int(ok)

        probs_by_cohort[cohort].append(float(pred.get("tool_prob") or 0.0))
        margins_by_cohort[cohort].append(float(pred.get("tool_margin") or 0.0))
        if pred_name and not ok:
            wrong_pairs[(gt_name, pred_name)] += 1

    print("=== Step 1a: Native Multi-Tool Accuracy (len(s.tools) >= 2) ===")
    tot_c = sum(v[0] for v in native_stats.values())
    tot_n = sum(v[1] for v in native_stats.values())
    print(f"  Overall native multi-tool: {tot_c / max(1, tot_n):.4f} ({tot_c}/{tot_n})")
    for bk in ("F_func", "A_clean", "other"):
        c, n = native_stats[bk]
        if n:
            print(f"    {bk:<8}: {c / n:.4f} ({c}/{n})")
    print("  By native candidate count T:", {k: f"{v[0]}/{v[1]} ({v[0]/v[1]:.3f})" for k, v in sorted(by_t.items())})

    print("\n=== Step 1b: Single-Tool + Random Pool Distractors (T=3) ===")
    if aug_stats[1]:
        print(f"  Single+random distractors: {aug_stats[0] / aug_stats[1]:.4f} ({aug_stats[0]}/{aug_stats[1]})")

    print("\n=== Step 1c: Score & Margin Separation (F_func vs A_clean) ===")
    for bk in ("F_func", "A_clean"):
        ps = probs_by_cohort.get(bk, [])
        ms = margins_by_cohort.get(bk, [])
        if ps:
            print(f"  {bk:<8} (n={len(ps):>3}) | tool_prob p10={_pct(ps,0.1):.3f} p50={_pct(ps,0.5):.3f} p90={_pct(ps,0.9):.3f} "
                  f"| margin p10={_pct(ms,0.1):.3f} p50={_pct(ms,0.5):.3f} p90={_pct(ms,0.9):.3f}")
    if wrong_pairs:
        print("  Top confusions (gt -> pred):", [(f"{g} -> {p}", cnt) for (g, p), cnt in wrong_pairs.most_common(6)])

eval_tool_prediction_step1(CLEAN_EVAL_PATH, os.path.join(DATA_DIR, f"{EVAL_SPLIT}.jsonl"))

=== Step 1a: Native Multi-Tool Accuracy (len(s.tools) >= 2) ===
  Overall native multi-tool: 0.9251 (247/267)
    F_func  : 0.9898 (195/197)
    A_clean : 1.0000 (35/35)
    other   : 0.4857 (17/35)
  By native candidate count T: {'2': '218/234 (0.932)', '3': '15/17 (0.882)', '4': '12/14 (0.857)', '5+': '2/2 (1.000)'}

=== Step 1b: Single-Tool + Random Pool Distractors (T=3) ===
  Single+random distractors: 0.9595 (142/148)

=== Step 1c: Score & Margin Separation (F_func vs A_clean) ===
  F_func   (n=209) | tool_prob p10=1.000 p50=1.000 p90=1.000 | margin p10=0.999 p50=1.000 p90=1.000
  A_clean  (n=111) | tool_prob p10=0.985 p50=1.000 p90=1.000 | margin p10=0.970 p50=1.000 p90=1.000
  Top confusions (gt -> pred): [('set_blind_openness -> schedule_blind_adjustment', 1), ('batch_classify_sentiments -> load_sentiment_model', 1), ('json_output -> simulate_arbitrage', 1)]


---
### 4b. Step 2 in Isolation — Parameter Extraction (`H3`, Oracle Tool Given)

Breaks Step 2 (`resolver.try_resolve(gt_tool, ...)`) into:
- **Step 2a**: Candidate Mask Parity (`T=all` vs `T=1`).
- **Step 2b**: Oracle Candidates (`protect_parity_spans=True`, gates OFF) vs Served Candidates (`protect_parity_spans=False`).
- **Step 2c**: Gated & Schema-Validated Extraction (`collapse_guard=True`, `min_joint_margin`).

In [10]:
import math
from spancall.lean.eval_p1 import eval_param_extraction_lean
from spancall.lean.resolver import LeanResolver

def _fmt_p1(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "n/a"
    return f"{v:.4f}" if isinstance(v, float) else str(v)

_p1_2a_all = eval_param_extraction_lean(
    CLEAN_EVAL_PATH, resolver=resolver, data_dir=DATA_DIR, limit=None,
    measure_candidate_parity=True, protect_parity_spans=False,
    context_mode="all", stratum_filter="F", exclude_tools=["json_output"], quiet=True,
)
_p1_2a_t1 = eval_param_extraction_lean(
    CLEAN_EVAL_PATH, resolver=resolver, data_dir=DATA_DIR, limit=None,
    measure_candidate_parity=True, protect_parity_spans=False,
    context_mode="single", stratum_filter="F", exclude_tools=["json_output"], quiet=True,
)
print("=== Step 2a: Candidate Mask Parity (F_func, protection OFF) ===")
print(f"  T=all (multi-tool render): parity={_fmt_p1(_p1_2a_all['candidate_parity'])} "
      f"({_p1_2a_all['candidate_parity_hit']}/{_p1_2a_all['candidate_parity_checked']}) | "
      f"miss_reasons={_p1_2a_all['parity_miss_reasons']}")
print(f"  T=1   (two-step render)  : parity={_fmt_p1(_p1_2a_t1['candidate_parity'])} "
      f"({_p1_2a_t1['candidate_parity_hit']}/{_p1_2a_t1['candidate_parity_checked']}) | "
      f"miss_reasons={_p1_2a_t1['parity_miss_reasons']}")

_step2_arms = [
    ("2b. Oracle Cands (protect=T, gates=OFF)",    True,  False, 0.0,              "single"),
    ("2a+2b. Served Cands (protect=F, gates=OFF)", False, False, 0.0,              "single"),
    (f"2c. + Guards & Margin ({MIN_JOINT_MARGIN}) [T=1]",  False, True, MIN_JOINT_MARGIN, "single"),
    (f"2c-all. + Guards & Margin ({MIN_JOINT_MARGIN}) [T=all]", False, True, MIN_JOINT_MARGIN, "all"),
]

print("\n=== Step 2b & 2c: Sub-Stage Waterfall (F_func vs A_clean) ===")
print(f"{'Stage / Config':<44}{'Parity':>8}{'Prec':>8}{'Rec':>8}{'F1':>8}{'Coll_F':>8}{'Exact_F':>9}{'A_Hit':>8}")
print("-" * 101)
for _label, _prot, _cg, _mjm, _cmode in _step2_arms:
    _r_arm = LeanResolver(
        model=model, encoder=encoder, tokenizer=tokenizer,
        max_seq_len=cfg.model.max_seq_len, max_span_width=cfg.model.max_span_width,
        max_calls=MAX_CALLS, call_min_prob=CALL_MIN_PROB, call_min_prob_ratio=CALL_MIN_PROB_RATIO,
        min_joint_margin=_mjm, collapse_guard=_cg, prevent_param_collapse=_cg,
    )
    _mf = eval_param_extraction_lean(
        CLEAN_EVAL_PATH, resolver=_r_arm, data_dir=DATA_DIR, limit=None,
        measure_candidate_parity=True, protect_parity_spans=_prot,
        context_mode=_cmode, stratum_filter="F", exclude_tools=["json_output"], quiet=True,
    )
    _ma = eval_param_extraction_lean(
        CLEAN_EVAL_PATH, resolver=_r_arm, data_dir=DATA_DIR, limit=None,
        measure_candidate_parity=False, protect_parity_spans=False,
        context_mode=_cmode, stratum_filter="A", exclude_tools=["json_output"], quiet=True,
    )
    _a_hit = (_ma["extracted"] / _ma["n"]) if _ma["n"] else float("nan")
    print(f"{_label:<44}{_fmt_p1(_mf['candidate_parity']):>8}"
          f"{_fmt_p1(_mf['extraction_precision']):>8}{_fmt_p1(_mf['extraction_recall']):>8}"
          f"{_fmt_p1(_mf['extraction_f1']):>8}{_fmt_p1(_mf['pointer_collapse_rate_F']):>8}"
          f"{_fmt_p1(_mf['exact_payload_match_F']):>9}{_fmt_p1(_a_hit):>8}")

=== Step 2a: Candidate Mask Parity (F_func, protection OFF) ===
  T=all (multi-tool render): parity=0.2162 (219/1013) | miss_reasons={'never_generated_typefilter': 201, 'truncated': 580, 'never_generated_width': 13}
  T=1   (two-step render)  : parity=0.8776 (889/1013) | miss_reasons={'truncated': 62, 'never_generated_typefilter': 55, 'never_generated_width': 7}

=== Step 2b & 2c: Sub-Stage Waterfall (F_func vs A_clean) ===
Stage / Config                                Parity    Prec     Rec      F1  Coll_F  Exact_F   A_Hit
-----------------------------------------------------------------------------------------------------
2b. Oracle Cands (protect=T, gates=OFF)       0.9220  0.6976  0.9968  0.8208  0.0526   0.5502  0.7444
2a+2b. Served Cands (protect=F, gates=OFF)    0.8776  0.6799  0.9968  0.8084  0.0766   0.5167  0.7444
2c. + Guards & Margin (0.02) [T=1]            0.8776  0.6907  0.9653  0.8053  0.0144   0.5167  0.7444
2c-all. + Guards & Margin (0.02) [T=all]      0.2162  0.6823  

---
### 4c. Step 3 — Composed E2E (`single_pass` vs `two_step`), Stage Waterfall & Headline Table

Runs E2E inference on `SUBSET_EVAL_PATH` in both `single_pass` and `two_step` modes, prints the stage-by-stage waterfall funnel, and compares `B1a (regex)` vs `SpanCall Lean (single_pass)` vs `SpanCall Lean (two_step)` via `scripts/eval_toolcall.py`.

In [11]:
import json, os, subprocess, sys
from typing import Any, Dict, List, Optional
from spancall.data.schema import UnifiedSample
import spancall.data.flatten as _flat
from spancall.eval.toolcall.evaluator import payload_equal
from spancall.lean.eval_p1 import _parity_spans_for_call, classify_eval_cohort, predicted_call_count as _n_predicted_calls

def _param_detail(pr) -> Dict[str, Any]:
    return {
        "value": getattr(pr, "value", None),
        "confidence": float(getattr(pr, "confidence", 0.0) or 0.0),
        "margin": float(getattr(pr, "margin", 0.0) or 0.0),
        "source_type": getattr(pr, "source_type", None),
        "char_span": list(pr.char_span) if getattr(pr, "char_span", None) else None,
        "p_abstain": 0.0,
    }

def _features_detail(features) -> Dict[str, Any]:
    feats = dict(features or {})
    if "tool_scores" in feats and isinstance(feats["tool_scores"], dict):
        top = sorted(feats["tool_scores"].items(), key=lambda kv: -float(kv[1]))[:3]
        feats["tool_scores"] = {k: float(v) for k, v in top}
    return {k: v for k, v in feats.items() if k not in ("recent_selected", "tool_scores_all")}

def predict_split(split_name: str, out_path: str, limit: Optional[int] = None,
                  detailed: bool = True, protect_parity_spans: bool = False,
                  resolve_mode: Optional[str] = None, src_path: Optional[str] = None) -> None:
    sub_src = os.path.join(RESULTS_DIR, f"subset_{split_name}.jsonl")
    src = src_path or (sub_src if (split_name == EVAL_SPLIT and os.path.exists(sub_src)) else os.path.join(DATA_DIR, f"{split_name}.jsonl"))
    if not os.path.exists(src):
        print(f"no source {src} — skipped")
        return
    preds: List[Dict[str, Any]] = []
    subset_samples: List[Dict[str, Any]] = []
    n = 0
    parity_checked = 0
    parity_hit = 0
    collapse_hits = 0
    _supports_parity = bool(getattr(resolver, "supports_candidate_parity", False))
    with open(src) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            raw_dict = json.loads(line)
            s = UnifiedSample.from_dict(raw_dict)
            _p_spans = None
            if _supports_parity and s.ground_truth_calls:
                _gt0 = s.ground_truth_calls[0]
                _raw_ps = _parity_spans_for_call(s, 0)
                if _raw_ps:
                    _p_spans = {f"{_gt0.tool_name}::{k}": v for k, v in _raw_ps.items()}
            if _supports_parity:
                res = resolver.try_resolve_candidates(
                    s.tools, s.utterance, s.recent_context, s.current_time_iso,
                    parity_spans=_p_spans, protect_parity_spans=protect_parity_spans,
                    resolve_mode=resolve_mode,
                )
            else:
                res = resolver.try_resolve_candidates(s.tools, s.utterance, s.recent_context, s.current_time_iso)
            _feats = getattr(res, "features", None) or {}
            parity_checked += int(_feats.get("parity_checked", 0) or 0)
            parity_hit += int(_feats.get("parity_hit", 0) or 0)
            if res.hit and _feats.get("pointer_collapsed"):
                collapse_hits += 1
            rec = {
                "sample_id": s.sample_id,
                "stratum": s.stratum.value,
                "gt_tool": (s.ground_truth_calls[0].tool_name if s.ground_truth_calls else None),
                "gt_args": (s.ground_truth_calls[0].arguments if s.ground_truth_calls else {}),
                "gt_calls": [{"tool_name": c.tool_name, "arguments": dict(c.arguments or {})}
                             for c in (s.ground_truth_calls or [])],
                "n_gt_calls": len(s.ground_truth_calls or []),
                "calls": [c.as_dict() for c in (getattr(res, "calls", None) or [])],
                "n_pred_calls": _n_predicted_calls(res),
                "hit": bool(res.hit),
                "status": getattr(res, "status", "hit" if res.hit else "partial"),
                "payload": res.payload,
                "payload_partial": getattr(res, "payload_partial", {}),
                "missing_required": getattr(res, "missing_required", []),
                "missing_optional": getattr(res, "missing_optional", []),
                "tool_name": res.tool_name,
                "confidence": float(getattr(res, "confidence", res.accept_score) or 0.0),
                "accept_probability": float(res.accept_score or 0.0),
                "joint_margin": float(getattr(res, "joint_margin", 0.0) or 0.0),
                "abstain_reason": res.abstain_reason,
                "latency_ms": float(getattr(res, "latency_ms", 0.0) or 0.0),
            }
            if detailed:
                rec["parameters"] = {p: _param_detail(pr) for p, pr in (res.parameters or {}).items()}
                rec["features"] = _features_detail(getattr(res, "features", {}))
            preds.append(rec)
            subset_samples.append(raw_dict)
            n += 1
            if limit is not None and n >= limit:
                break
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as fh:
        for p in preds:
            fh.write(json.dumps(p, default=str) + "\n")
    if os.path.abspath(src) != os.path.abspath(sub_src):
        with open(sub_src, "w") as fh:
            for s_dict in subset_samples:
                fh.write(json.dumps(s_dict, default=str) + "\n")
    n_hit = sum(1 for p in preds if p["hit"])
    _par_str = f"{parity_hit}/{parity_checked} ({parity_hit / parity_checked:.3f})" if parity_checked else "n/a"
    print(f"predicted {n} rows ({resolve_mode or 'default'}) -> {out_path} (hit={n_hit}, collapse_hits={collapse_hits}, e2e_parity={_par_str})")

_p_single = os.path.join(RESULTS_DIR, "preds_eval.jsonl")
_p_two    = os.path.join(RESULTS_DIR, "preds_eval_twostep.jsonl")
predict_split(EVAL_SPLIT, _p_single, limit=None, resolve_mode="single_pass", src_path=SUBSET_EVAL_PATH)
predict_split(EVAL_SPLIT, _p_two,    limit=None, resolve_mode="two_step",    src_path=SUBSET_EVAL_PATH)

_raw_sub = {}
with open(SUBSET_EVAL_PATH) as _fh:
    for _l in _fh:
        if _l.strip():
            _d = json.loads(_l)
            _raw_sub[_d["sample_id"]] = (_d, UnifiedSample.from_dict(_d))

def _waterfall(preds_path: str, label: str) -> None:
    with open(preds_path) as _fh:
        _ps = [json.loads(l) for l in _fh if l.strip()]
    _f  = [p for p in _ps if classify_eval_cohort(_raw_sub[p["sample_id"]][0]) == "F_func"]
    _ac = [p for p in _ps if classify_eval_cohort(_raw_sub[p["sample_id"]][0]) == "A_clean"]
    _aa = [p for p in _ps if classify_eval_cohort(_raw_sub[p["sample_id"]][0]) == "A_adv"]

    n_tot = len(_f)
    s0 = [p for p in _f if not str(p.get("abstain_reason") or "").startswith("sanitize")]
    s1 = [p for p in s0 if p.get("tool_name") == p.get("gt_tool")]
    s2a = [p for p in s1 if int((p.get("features") or {}).get("parity_checked") or 0) == int((p.get("features") or {}).get("parity_hit") or 0)]
    s2b_exact = []
    for p in s2a:
        _s = _raw_sub[p["sample_id"]][1]
        _gt_t = next((t for t in _s.tools if t.name == p["gt_tool"]), None)
        # v2.7: GT args are NESTED while ``payload_partial`` carries dotted
        # leaf keys. Comparing them raw makes every object leaf a mismatch,
        # which drops the whole waterfall one stage early. flatten_args is
        # idempotent, so this holds before and after the resolver unflattens.
        _paths = dict(getattr(_gt_t, "object_paths", None) or {})
        _gt_flat = _flat.flatten_args(p.get("gt_args") or {}, _paths)
        _pred_flat = _flat.flatten_args(p.get("payload_partial") or {}, _paths)
        if payload_equal(_gt_flat, _pred_flat, _gt_t):
            s2b_exact.append(p)
    s2c_hit = [p for p in s2b_exact if p.get("hit")]

    print(f"\n--- Waterfall Funnel [{label}] on F_func (N={n_tot}) ---")
    for _name, _cnt in [
        ("Total F_func", n_tot),
        ("-> Step 0: Sanitize OK", len(s0)),
        ("-> Step 1: Correct Tool", len(s1)),
        ("-> Step 2a: All GT Spans in C=512", len(s2a)),
        ("-> Step 2b: All Params Matched (partial)", len(s2b_exact)),
        ("-> Step 2c: Schema & Gates Hit (E2E Exact)", len(s2c_hit)),
    ]:
        print(f"  {_name:<44} {_cnt:>4} / {n_tot} ({_cnt / max(1, n_tot):>6.1%})")
    _ac_hit = sum(1 for p in _ac if p.get("hit"))
    _aa_san = sum(1 for p in _aa if str(p.get("abstain_reason") or "").startswith("sanitize"))
    print(f"  [A_clean] correct abstain: {len(_ac) - _ac_hit}/{len(_ac)} ({1.0 - _ac_hit / max(1, len(_ac)):.1%}) | "
          f"[A_adv] sanitize-vetoed: {_aa_san}/{len(_aa)} ({_aa_san / max(1, len(_aa)):.1%})")

_waterfall(_p_single, "single_pass")
_waterfall(_p_two, "two_step")

# Evaluate B1a, single_pass, and two_step via eval_toolcall.py
for _args, _out in [
    (["--baseline", "b1a"], f"toolcall_metrics_{EVAL_SPLIT}_b1a.json"),
    (["--predictions", _p_single], f"toolcall_metrics_{EVAL_SPLIT}_lean.json"),
    (["--predictions", _p_two], f"toolcall_metrics_{EVAL_SPLIT}_lean_twostep.json"),
]:
    subprocess.run([
        sys.executable, "scripts/eval_toolcall.py",
        "--test_data", SUBSET_EVAL_PATH,
        *_args,
        "--output", os.path.join(RESULTS_DIR, _out),
    ], check=True)

def _load_metrics(path):
    if not os.path.exists(path):
        return None
    with open(path) as fh:
        return json.load(fh)

print("\n=== Headline Comparison Table (hermes_eval) ===")
_headers = ["system", "n_F", "exact_F", "set_exact_F", "count_acc", "hit_F", "collapse_F", "param_F1_F", "false_hit_on_A", "exact_all"]
print(" | ".join(_headers))
print("-|" * len(_headers))
for _label, _fn in [
    ("B1a (regex)", f"toolcall_metrics_{EVAL_SPLIT}_b1a.json"),
    ("SpanCall Lean (single_pass)", f"toolcall_metrics_{EVAL_SPLIT}_lean.json"),
    ("SpanCall Lean (two_step)", f"toolcall_metrics_{EVAL_SPLIT}_lean_twostep.json"),
]:
    _m = _load_metrics(os.path.join(RESULTS_DIR, _fn))
    if not _m:
        continue
    _mf = _m.get("metrics_by_stratum", {}).get("F", {})
    _row = {
        "system": _label,
        "n_F": int(_mf.get("n", 0)),
        "exact_F": round(_mf.get("exact_payload_match", 0.0), 3),
        "set_exact_F": round(_mf.get("call_exact_set_match", 0.0), 3),
        "count_acc": round(_m.get("call_count_acc", 0.0), 3),
        "hit_F": round(_mf.get("hit_rate", 0.0), 3),
        "collapse_F": round(_mf.get("collapse_hit_rate", 0.0), 3),
        "param_F1_F": round(_mf.get("param_f1", 0.0), 3),
        "false_hit_on_A": round(_m.get("false_hit_rate_on_a", 0.0), 4),
        "exact_all": round(_m.get("exact_payload_match", 0.0), 3),
    }
    print(" | ".join(str(_row[h]) for h in _headers))

predicted 415 rows (single_pass) -> /content/spancall/results_lean_v2_7/preds_eval.jsonl (hit=289, collapse_hits=0, e2e_parity=611/1565 (0.390))
predicted 415 rows (two_step) -> /content/spancall/results_lean_v2_7/preds_eval_twostep.jsonl (hit=287, collapse_hits=0, e2e_parity=1312/1565 (0.838))

--- Waterfall Funnel [single_pass] on F_func (N=209) ---
  Total F_func                                  209 / 209 (100.0%)
  -> Step 0: Sanitize OK                        209 / 209 (100.0%)
  -> Step 1: Correct Tool                       207 / 209 ( 99.0%)
  -> Step 2a: All GT Spans in C=512              33 / 209 ( 15.8%)
  -> Step 2b: All Params Matched (partial)       23 / 209 ( 11.0%)
  -> Step 2c: Schema & Gates Hit (E2E Exact)     17 / 209 (  8.1%)
  [A_clean] correct abstain: 73/111 (65.8%) | [A_adv] sanitize-vetoed: 23/45 (51.1%)

--- Waterfall Funnel [two_step] on F_func (N=209) ---
  Total F_func                                  209 / 209 (100.0%)
  -> Step 0: Sanitize OK             

---
### 4d. Object-Decomposition Audit

Quantifies the v2.7 change on the evaluation split:

- **Decomposition coverage** — of the `object` params the GT declares, how many exist in the schema as dotted leaves (`decomposed`) versus as a single opaque `object` param (`unresolved`, routed to the slow path). A schema-level number, independent of model quality.
- **Outcome split by param type** — exact-match rate per `string` / `integer` / `number` / `boolean` / `enum` / `array` / `object` / `object-leaf`. This is the table that shows whether decomposition moved objects and how large the remaining array gap is.
- **Obj-shaped residual examples** — up to 5 failing rows whose only mismatches are object leaves or arrays, with the leaf-level diff, so "did reassembly round-trip" is checkable by eye rather than inferred from a scalar.

In [12]:
# 4d. Object-decomposition audit (schema coverage + per-param-type accuracy + residual examples)
import json, os
from collections import Counter
from typing import Any, Dict, List

import spancall.data.flatten as _flat
from spancall.data.schema import UnifiedSample
from spancall.eval.toolcall.evaluator import _deep_equal


def _is_object_leaf(name: str, object_paths: Dict[str, List[str]]) -> bool:
    root = str(name).split(_flat.PATH_SEP, 1)[0]
    return bool(object_paths) and any(
        str(leaf) == str(name) for leaf in (object_paths.get(root) or ())
    )


def _gather_samples(path: str, limit: int = 0) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    if not os.path.exists(path):
        print(f"no source {path} - skipped")
        return rows
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
            if limit and len(rows) >= limit:
                break
    return rows


# ---------------------------------------------------------------- schema view
# This is a SCHEMA-level number, so it must be counted once per distinct tool.
# The same tool schema repeats in every sample that offers it (~10-50x), and a
# per-occurrence counter would silently turn "tools with decomposed objects"
# into a sample-weighted count - a number that can exceed the number of distinct
# tools in the split and moves when the eval slice size changes.
_schema_stats = Counter()
_seen_tools: set = set()
_dup_tools = 0
for _raw in _gather_samples(os.path.join(DATA_DIR, f"{EVAL_SPLIT}.jsonl")):
    _s = UnifiedSample.from_dict(_raw)
    for _t in _s.tools:
        if _t.name in _seen_tools:
            _dup_tools += 1
            continue
        _seen_tools.add(_t.name)
        _opaque = [p.name for p in _t.parameters if (p.param_type or "").lower() == "object"]
        _leaves = [p.name for p in _t.parameters if _is_object_leaf(p.name, _t.object_paths or {})]
        if _opaque:
            _schema_stats["tools_with_opaque_object"] += 1
            _schema_stats["opaque_object_params"] += len(_opaque)
        if _leaves:
            _schema_stats["tools_with_decomposed_object"] += 1
            _schema_stats["decomposed_leaf_params"] += len(_leaves)
        if _t.object_paths:
            _schema_stats["object_roots_decomposed"] += len(_t.object_paths)

print("=== Object Decomposition Coverage (schema level, distinct tools) ===")
print(f"  distinct tool schemas scanned    : {len(_seen_tools)}"
      f"  ({_dup_tools} repeated occurrences ignored)")
print(f"  object roots decomposed          : {_schema_stats['object_roots_decomposed']}")
print(f"  scalar leaf params emitted       : {_schema_stats['decomposed_leaf_params']}")
print(f"  tools with decomposed objects    : {_schema_stats['tools_with_decomposed_object']}")
print(f"  tools still carrying an opaque object param : {_schema_stats['tools_with_opaque_object']}")
print(f"  opaque object params (routed to slow path)  : {_schema_stats['opaque_object_params']}")
if not _schema_stats["object_roots_decomposed"]:
    print("  [i] no decomposable object found in this split - decomposition is a no-op here.")

# ------------------------------------------------------------ per-type outcome
_PRED_PATH = os.path.join(RESULTS_DIR, "preds_eval.jsonl")
_preds_by_id: Dict[str, Dict[str, Any]] = {}
for _p in _gather_samples(_PRED_PATH):
    _preds_by_id[_p["sample_id"]] = _p

_type_tot = Counter()
_type_ok = Counter()
_residual: List[Dict[str, Any]] = []

if not _preds_by_id:
    print(f"\n[!] no predictions at {_PRED_PATH} - run 4c first. Per-type table skipped.")

for _raw in _gather_samples(SUBSET_EVAL_PATH):
    _sid = _raw.get("sample_id")
    _p = _preds_by_id.get(_sid)
    if _p is None or not _p.get("gt_args"):
        continue
    _s = UnifiedSample.from_dict(_raw)
    if not _s.ground_truth_calls:
        continue
    _gt_call = _s.ground_truth_calls[0]
    _gt_tool = next((t for t in _s.tools if t.name == _gt_call.tool_name), None)
    if _gt_tool is None:
        continue
    _paths = dict(_gt_tool.object_paths or {})
    _gt_full = _flat.flatten_args(_gt_call.arguments or {}, _paths)

    # The predicted payload must be flattened into the SAME key space as the GT,
    # otherwise a nested ``{"filters": {...}}`` never matches a ``filters.color``
    # slot and every object leaf scores as wrong regardless of model quality.
    # ``flatten_args`` is idempotent for already-flat payloads, so this is safe
    # both before and after the resolver's unflatten step lands.
    _pred_raw = _p.get("payload_partial") or {}
    _pred_args = _flat.flatten_args(_pred_raw, _paths)

    _bad_names: List[str] = []

    for _param in _gt_tool.parameters:
        _name = _param.name
        if _name not in _gt_full:
            # Not present in this call's GT arguments -> nothing to score.
            continue
        if _is_object_leaf(_name, _paths):
            _kind = "object-leaf"
        else:
            _kind = (_param.param_type or "string").lower()
        _type_tot[_kind] += 1
        _ok = _name in _pred_args and _deep_equal(_gt_full[_name], _pred_args[_name])
        if _ok:
            _type_ok[_kind] += 1
        else:
            _bad_names.append(_name)

    # Residual rows whose failures are ONLY object leaves / arrays.
    if _bad_names:
        _kinds = set()
        for _n in _bad_names:
            _pp = next((q for q in _gt_tool.parameters if q.name == _n), None)
            _kinds.add("object-leaf" if _is_object_leaf(_n, _paths)
                       else ((_pp.param_type or "string").lower() if _pp else "unknown"))
        if _kinds <= {"object-leaf", "array", "object"}:
            _residual.append({"sample_id": _sid, "tool": _gt_call.tool_name,
                              "bad": _bad_names, "kinds": sorted(_kinds),
                              "gt": _gt_full, "pred": _pred_args})

print("\n=== Per-Param-Type Exact Match (extractable params, hermes_eval) ===")
print(f"{'param_type':<14}{'n':>6}{'exact':>9}{'rate':>8}")
print("-" * 40)
for _k in sorted(_type_tot, key=lambda x: -_type_tot[x]):
    _n, _ok = _type_tot[_k], _type_ok[_k]
    _rate = f"{_ok / _n:>8.3f}" if _n else f"{'n/a':>8}"
    print(f"{_k:<14}{_n:>6}{_ok:>9}{_rate}")

_n_obj = _type_tot["object-leaf"]
print(f"\n  object-leaf params scored: {_n_obj}"
      f"  (exact {_type_ok['object-leaf']}/{_n_obj})")
print(f"  array params scored      : {_type_tot['array']}"
      f"  (exact {_type_ok['array']}/{_type_tot['array']})")
if not _n_obj:
    print("  [!] zero object-leaf params scored: either no object-bearing tool is in")
    print("      this split, or step 3 (aligner object case) is not applied yet.")
print("  [i] 'object-leaf' is the decomposed form: a scalar slot that reassembles into")
print("      the parent object. A rate here above the pre-v2.7 'object' rate is the win.")

print(f"\n=== Residual Rows Failing Only On Complex Params ({len(_residual)} total; showing up to 5) ===")
for _r in _residual[:5]:
    print("-" * 100)
    print(f"{_r['sample_id']}  tool={_r['tool']}  kinds={_r['kinds']}")
    for _n in _r["bad"][:6]:
        print(f"  [WRONG] {_n:<28} gt={_r['gt'].get(_n)!r:<42} pred={_r['pred'].get(_n)!r}")
if not _residual:
    print("  none - no row failed exclusively on object/array params.")

# Reassembly integrity: does the emitted payload equal the unflattened GT exactly?
# Reassembly integrity is checked on the LEAVES, not on the whole payload.
# ``payload_equal`` requires identical key sets, so using it here would report
# "reassembly broken" whenever the model merely omitted an argument -- mixing a
# model-quality signal into a correctness check. What this block must prove is
# narrower and model-independent: for every object leaf the model DID emit, does
# unflatten sit that leaf back at the same dotted path with the same value?
_rt_tot = _rt_ok = 0
_rt_bad: List[Dict[str, Any]] = []
for _raw in _gather_samples(SUBSET_EVAL_PATH):
    _p = _preds_by_id.get(_raw.get("sample_id"))
    if _p is None:
        continue
    _s = UnifiedSample.from_dict(_raw)
    if not _s.ground_truth_calls:
        continue
    _gt_tool = next((t for t in _s.tools if t.name == _s.ground_truth_calls[0].tool_name), None)
    if _gt_tool is None or not _gt_tool.object_paths:
        continue
    _rt_tot += 1
    # Normalise to the flat key space first: ``get_path`` walks dotted segments,
    # so on a flat payload it would always report ``_MISSING``. The resolver's
    # payload may be flat (pre-step-5) or nested (post-step-5); flattening both
    # forms idempotently makes this check true in either case.
    _pred_flat = _flat.flatten_args(_p.get("payload_partial") or {},
                                    _gt_tool.object_paths)
    _rebuilt = _flat.unflatten_args(_pred_flat, _gt_tool.object_paths, strict=False)
    # Walk every declared leaf: does it survive the flat -> nested round trip?
    _leaf_bad = []
    for _root, _leaves in _gt_tool.object_paths.items():
        for _leaf in _leaves:
            _before = _pred_flat.get(str(_leaf), _flat.MISSING)
            _after = _flat.get_path(_rebuilt, _leaf)
            if _before is _flat.MISSING and _after is _flat.MISSING:
                continue  # not emitted and not reassembled -> consistent
            if _before is _flat.MISSING or not _deep_equal(_before, _after):
                _leaf_bad.append({"leaf": _leaf,
                                  "flat": None if _before is _flat.MISSING else _before,
                                  "nested": None if _after is _flat.MISSING else _after})
    if _leaf_bad:
        _rt_bad.append({"sample_id": _raw.get("sample_id"), "leaves": _leaf_bad})
    else:
        _rt_ok += 1

print("\n=== Reassembly Integrity (object-bearing calls) ===")
if _rt_tot:
    print(f"  calls with decomposed objects        : {_rt_tot}")
    print(f"  every emitted leaf reassembles at its path : {_rt_ok} ({_rt_ok / _rt_tot:.3f})")
    for _b in _rt_bad[:3]:
        print(f"  [LEAF MISMATCH] {_b['sample_id']}: {_b['leaves'][:3]}")
    if not _rt_bad:
        print("  no leaf-level reassembly defect - flat/nested round trip is sound.")
else:
    print("  no object-bearing call in this split")

=== Object Decomposition Coverage (schema level, distinct tools) ===
  distinct tool schemas scanned    : 2602  (3739 repeated occurrences ignored)
  object roots decomposed          : 0
  scalar leaf params emitted       : 0
  tools with decomposed objects    : 0
  tools still carrying an opaque object param : 234
  opaque object params (routed to slow path)  : 254
  [i] no decomposable object found in this split - decomposition is a no-op here.

=== Per-Param-Type Exact Match (extractable params, hermes_eval) ===
param_type         n    exact    rate
----------------------------------------
string           679      480   0.707
number           107       80   0.748
array             87       10   0.115
object            78        0   0.000
integer           76       62   0.816
boolean           17       14   0.824
['string', 'null']     2        0   0.000

  object-leaf params scored: 0  (exact 0/0)
  array params scored      : 87  (exact 10/87)
  [!] zero object-leaf params scored: 

---
### 4e. Stage-Attributed Error Forensics

Groups `F_func` misses by the exact stage where they failed (`Step 1 WRONG_TOOL`, `Step 2a PARITY_MISS`, `Step 2b POINTER_RANK_MISS`, `Step 2c GATE_OR_SCHEMA_REJECT`).

In [13]:
import json, os, textwrap
from collections import defaultdict
from spancall.data.schema import UnifiedSample
import spancall.data.flatten as _flat
from spancall.eval.toolcall.evaluator import payload_equal
from spancall.lean.eval_p1 import classify_eval_cohort

PREDS_PATH = os.path.join(RESULTS_DIR, "preds_eval_twostep.jsonl")
if not os.path.exists(PREDS_PATH):
    PREDS_PATH = os.path.join(RESULTS_DIR, "preds_eval.jsonl")

if os.path.exists(PREDS_PATH) and os.path.exists(SUBSET_EVAL_PATH):
    with open(PREDS_PATH) as fh:
        preds = [json.loads(l) for l in fh if l.strip()]
    subs, raw_map = {}, {}
    with open(SUBSET_EVAL_PATH) as fh:
        for l in fh:
            if l.strip():
                d = json.loads(l)
                raw_map[d["sample_id"]] = d
                subs[d["sample_id"]] = UnifiedSample.from_dict(d)

    buckets = defaultdict(list)
    for p in preds:
        sid = p.get("sample_id")
        if sid not in raw_map or classify_eval_cohort(raw_map[sid]) != "F_func":
            continue
        feats = p.get("features") or {}
        if str(p.get("abstain_reason") or "").startswith("sanitize"):
            buckets["STEP0_SANITIZE"].append(p)
        elif p.get("tool_name") != p.get("gt_tool"):
            buckets["STEP1_WRONG_TOOL"].append(p)
        elif int(feats.get("parity_hit") or 0) < int(feats.get("parity_checked") or 0):
            buckets["STEP2A_PARITY_MISS"].append(p)
        else:
            s = subs[sid]
            gt_t = next((t for t in s.tools if t.name == p["gt_tool"]), None)
            # v2.7: both sides flattened into the declared object key space.
            # A nested GT vs dotted prediction would otherwise attribute a
            # reassembly-correct row to STEP2B (pointer rank miss).
            _paths = dict(getattr(gt_t, "object_paths", None) or {})
            _gt_flat = _flat.flatten_args(p.get("gt_args") or {}, _paths)
            _part_flat = _flat.flatten_args(p.get("payload_partial") or {}, _paths)
            partial_ok = payload_equal(_gt_flat, _part_flat, gt_t)
            if p.get("hit") and payload_equal(
                _gt_flat, _flat.flatten_args(p.get("payload") or {}, _paths), gt_t
            ):
                buckets["EXACT_HIT"].append(p)
            elif not partial_ok:
                buckets["STEP2B_POINTER_RANK_MISS"].append(p)
            else:
                buckets["STEP2C_GATE_OR_SCHEMA_REJECT"].append(p)

    tot_f = sum(len(v) for v in buckets.values())
    print(f"F_func Stage Attribution ({os.path.basename(PREDS_PATH)}, N={tot_f}):")
    for k in ("EXACT_HIT", "STEP0_SANITIZE", "STEP1_WRONG_TOOL", "STEP2A_PARITY_MISS", "STEP2B_POINTER_RANK_MISS", "STEP2C_GATE_OR_SCHEMA_REJECT"):
        c = len(buckets[k])
        print(f"  {k:<30}{c:>4}  ({c / max(1, tot_f):>6.1%})")

F_func Stage Attribution (preds_eval_twostep.jsonl, N=209):
  EXACT_HIT                      108  ( 51.7%)
  STEP0_SANITIZE                   0  (  0.0%)
  STEP1_WRONG_TOOL                 2  (  1.0%)
  STEP2A_PARITY_MISS              69  ( 33.0%)
  STEP2B_POINTER_RANK_MISS        24  ( 11.5%)
  STEP2C_GATE_OR_SCHEMA_REJECT     6  (  2.9%)


---
### 4f. Param-Precision FP Deep Dive (diagnostic)

Dumps `F_func` rows that reached Step 2 (right tool, parity OK) but produced a wrong payload, with per-parameter `confidence` / `margin` / `source_type` / `char_span` detail. Purpose: test the hypothesis that the decoder force-assigns argmax values to params whose value is absent or ambiguous (T6 refusal off: `min_confidence` / `min_posterior` are `None`, and the greedy fallback in `decode_detailed` returns top-1 with `delta=0`).

Also sweeps per-param confidence / margin thresholds offline over the stored `parameters` detail to estimate the precision/recall tradeoff of enabling per-param refusal.

In [14]:
import json, os
from collections import Counter, defaultdict
from spancall.data.schema import UnifiedSample
import spancall.data.flatten as _flat
from spancall.eval.toolcall.evaluator import payload_equal
from spancall.lean.eval_p1 import classify_eval_cohort

N_DUMP = 20  # number of FP rows to print in full

# v2.7: self-contained. This cell used to close over PREDS_PATH and
# raw_map left behind by 4e, so running or re-running it alone raised
# NameError; and a stale PREDS_PATH from an earlier run silently scored
# the wrong file. Recompute both here, same precedence as 4e.
PREDS_PATH = os.path.join(RESULTS_DIR, "preds_eval_twostep.jsonl")
if not os.path.exists(PREDS_PATH):
    PREDS_PATH = os.path.join(RESULTS_DIR, "preds_eval.jsonl")
if not os.path.exists(PREDS_PATH):
    raise SystemExit(f"no predictions at {PREDS_PATH} - run 4c first.")
raw_map, subs = {}, {}
if os.path.exists(SUBSET_EVAL_PATH):
    with open(SUBSET_EVAL_PATH) as _fh:
        for _l in _fh:
            if not _l.strip():
                continue
            _d = json.loads(_l)
            raw_map[_d["sample_id"]] = _d
            subs[_d["sample_id"]] = UnifiedSample.from_dict(_d)
else:
    print(f"[!] no eval subset at {SUBSET_EVAL_PATH} - cohort filter skipped.")

with open(PREDS_PATH) as fh:
    _preds = [json.loads(l) for l in fh if l.strip()]

fp_rows = []
fp_reason_counter = Counter()
conf_hist, margin_hist = [], []
delta_zero_rows = 0

for p in _preds:
    sid = p.get("sample_id")
    if sid not in raw_map or classify_eval_cohort(raw_map[sid]) != "F_func":
        continue
    feats = p.get("features") or {}
    if str(p.get("abstain_reason") or "").startswith("sanitize"):
        continue
    if p.get("tool_name") != p.get("gt_tool"):
        continue
    if int(feats.get("parity_hit") or 0) < int(feats.get("parity_checked") or 0):
        continue  # candidate-generation miss, not a precision problem
    # v2.7: pass the REAL tool schema (None skips optional-default
    # materialisation, so a correct row that simply omitted an optional
    # param was counted as a precision FP) and flatten both sides into the
    # declared object key space (nested GT vs dotted payload).
    _s = subs.get(sid)
    _gt_t = None
    if _s is not None:
        _gt_t = next((t for t in _s.tools if t.name == p.get("gt_tool")), None)
    _paths = dict(getattr(_gt_t, "object_paths", None) or {})
    _gt_flat = _flat.flatten_args(p.get("gt_args") or {}, _paths)
    p["_gt_flat"] = _gt_flat  # flat GT, reused by the refusal sweep below
    if p.get("hit") and payload_equal(
        _gt_flat, _flat.flatten_args(p.get("payload") or {}, _paths), _gt_t
    ):
        continue
    # This row passed Step 0/1/2a but failed payload equality -> precision FP.
    if abs(float(p.get("joint_margin") or 0.0)) < 1e-9:
        delta_zero_rows += 1
    params = p.get("parameters") or {}
    gt_args = p.get("_gt_flat") or {}  # flat, so dotted leaves line up
    for pn, pr in params.items():
        conf_hist.append(float(pr.get("confidence") or 0.0))
        margin_hist.append(float(pr.get("margin") or 0.0))
        pv = pr.get("value")
        if pn not in gt_args:
            fp_reason_counter["param_not_in_gt"] += 1
        elif gt_args[pn] != pv and gt_args[pn] is not None:
            fp_reason_counter["wrong_value"] += 1
        elif gt_args[pn] is None and pv is not None:
            fp_reason_counter["should_be_null"] += 1
    fp_rows.append(p)

print(f"F_func Step-2 precision-FP rows: {len(fp_rows)} / {sum(1 for p in _preds if p.get('sample_id') in raw_map and classify_eval_cohort(raw_map[p['sample_id']]) == 'F_func')}")
print(f"  joint_margin == 0 (greedy fallback / no runner-up) rows: {delta_zero_rows}")
print(f"  FP reasons: {dict(fp_reason_counter)}")
if conf_hist:
    conf_hist.sort(); margin_hist.sort()
    def _q(a, q): return a[min(len(a) - 1, int(q * len(a)))]
    print(f"  predicted-param confidence: p10={_q(conf_hist, 0.10):.3f} p50={_q(conf_hist, 0.50):.3f} p90={_q(conf_hist, 0.90):.3f}")
    print(f"  predicted-param margin:      p10={_q(margin_hist, 0.10):.3f} p50={_q(margin_hist, 0.50):.3f} p90={_q(margin_hist, 0.90):.3f}")

for p in fp_rows[:N_DUMP]:
    print("-" * 100)
    print(f"{p['sample_id']}  tool={p.get('tool_name')}  hit={p.get('hit')}  "
          f"joint_margin={float(p.get('joint_margin') or 0.0):.4f}")
    print(f"  utterance: {(raw_map[p['sample_id']].get('utterance') or '')[:220]!r}")
    print(f"  gt_args:   {json.dumps(p.get('gt_args') or {}, default=str)[:400]}")
    print(f"  payload:   {json.dumps(p.get('payload') or {}, default=str)[:400]}")
    for pn, pr in (p.get("parameters") or {}).items():
        gt_v = (p.get("_gt_flat") or {}).get(pn, "<NOT_IN_GT>")
        flag = "OK" if gt_v == pr.get("value") else ("NULL?" if gt_v is None else "WRONG")
        print(f"    [{flag:<6}] {pn:<28} pred={str(pr.get('value'))[:60]!s:<62} gt={str(gt_v)[:40]:<42} conf={float(pr.get('confidence') or 0.0):.3f} margin={float(pr.get('margin') or 0.0):.3f} src={pr.get('source_type')} span={pr.get('char_span')}")

# Offline refusal sweep: what would per-param thresholding have done?
if conf_hist or margin_hist:
    print("\n=== Offline Per-Param Refusal Sweep (stored predictions, no re-inference) ===")
    print(f"{'conf>=':>7}{'margin>=':>9}{'dropped':>9}{'dropped_OK':>12}{'dropped_bad':>12}")
    for c_thr in (0.0, 0.3, 0.5, 0.7):
        for m_thr in (0.0, 0.05, 0.1, 0.2):
            dropped = ok = bad = 0
            for p in fp_rows:
                gt_args = p.get("_gt_flat") or {}
                for pn, pr in (p.get("parameters") or {}).items():
                    if float(pr.get("confidence") or 0.0) < c_thr or float(pr.get("margin") or 0.0) < m_thr:
                        dropped += 1
                        if pn in gt_args and gt_args[pn] == pr.get("value"):
                            ok += 1  # would have removed a CORRECT param
                        else:
                            bad += 1  # would have removed a wrong/spurious param
            print(f"{c_thr:>7.2f}{m_thr:>9.2f}{dropped:>9d}{ok:>12d}{bad:>12d}")
    print("(dropped_bad >> dropped_OK at the operating point => per-param refusal (T6) would recover precision)")

F_func Step-2 precision-FP rows: 30 / 209
  joint_margin == 0 (greedy fallback / no runner-up) rows: 6
  FP reasons: {'wrong_value': 26, 'param_not_in_gt': 7}
  predicted-param confidence: p10=0.156 p50=0.795 p90=0.999
  predicted-param margin:      p10=0.028 p50=0.677 p90=0.999
----------------------------------------------------------------------------------------------------
glaive_func_calling::8c19d166-505c-4645-ba5a-88b96c8116d2  tool=get_random_fact  hit=False  joint_margin=0.0000
  utterance: "That's really cool! I didn't know that. Thanks for sharing these facts with me."
  gt_args:   {}
  payload:   {}
----------------------------------------------------------------------------------------------------
glaive_func_calling::08e63ff4-ef5c-49e1-969b-9aa436747907  tool=convert_temperature  hit=True  joint_margin=0.7532
  utterance: "Thank you, that's all I needed to know."
  gt_args:   {"temperature": 98.6, "from_unit": "Fahrenheit", "to_unit": "Celsius"}
  payload:   {"from_unit"

---
## 5. OOD / Adversarial Test Splits & Drive Persistence

Evaluates `LeanResolver` and `B1a` on `hermes_test_ood_schema`, `hermes_test_ood_para`, and `hermes_test_adversarial`, and optionally mirrors artifacts to Google Drive.

In [15]:
import json, os, shutil, subprocess, sys
from spancall.lean.eval_p1 import build_stratified_eval_subset

for split in TEST_SPLITS:
    sub_test_path = os.path.join(RESULTS_DIR, f"subset_{split}.jsonl")
    build_stratified_eval_subset(
        split,
        data_dir=DATA_DIR,
        out_path=sub_test_path,
        limit=TEST_LIMIT,
        seed=SEED,
    )
    if SINGLE_CALL_ONLY:
        with open(sub_test_path, encoding="utf-8") as _fh:
            _one_call_lines = [
                _l for _l in _fh
                if _l.strip() and len((json.loads(_l).get("ground_truth_calls") or [])) == 1
            ]
        with open(sub_test_path, "w", encoding="utf-8") as _fh:
            _fh.writelines(_one_call_lines)
    predict_split(
        split,
        os.path.join(RESULTS_DIR, f"preds_{split}.jsonl"),
        limit=None,
        resolve_mode="two_step",
        src_path=sub_test_path,
    )
    data_file = sub_test_path
    for label, extra in [("b1a", ["--baseline", "b1a"]), ("lean", ["--predictions", os.path.join(RESULTS_DIR, f"preds_{split}.jsonl")])]:
        subprocess.run([
            sys.executable, "scripts/eval_toolcall.py",
            "--test_data", data_file,
            *extra,
            "--output", os.path.join(RESULTS_DIR, f"toolcall_metrics_{split}_{label}.json"),
        ], check=False)

print(f"\n{'split':<26}{'system':<12}{'n_F':>6}{'exact_F':>9}{'hit_F':>8}{'param_F1':>9}{'exact_all':>10}")
print("-" * 80)
for split in TEST_SPLITS + [EVAL_SPLIT]:
    for label in ("b1a", "lean"):
        m = _load_metrics(os.path.join(RESULTS_DIR, f"toolcall_metrics_{split}_{label}.json"))
        if not m:
            continue
        m_f = m.get("metrics_by_stratum", {}).get("F", {})
        print(f"{split:<26}{label:<12}{int(m_f.get('n', 0)):>6}"
              f"{m_f.get('exact_payload_match', 0.0):>9.3f}{m_f.get('hit_rate', 0.0):>8.3f}"
              f"{m_f.get('param_f1', 0.0):>9.3f}{m.get('exact_payload_match', 0.0):>10.3f}")

if DRIVE_MOUNT:
    os.makedirs(DRIVE_OUT, exist_ok=True)
    for name, src in [("data_tensorized_lean_v2_7", TENSOR_DIR), ("results_lean_v2_7", RESULTS_DIR)]:
        if os.path.isdir(src):
            dst = os.path.join(DRIVE_OUT, name)
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"mirrored {src} -> {dst}")

predicted 158 rows (two_step) -> /content/spancall/results_lean_v2_7/preds_hermes_test_ood_schema.jsonl (hit=119, collapse_hits=0, e2e_parity=537/609 (0.882))
predicted 177 rows (two_step) -> /content/spancall/results_lean_v2_7/preds_hermes_test_ood_para.jsonl (hit=147, collapse_hits=0, e2e_parity=778/912 (0.853))
predicted 170 rows (two_step) -> /content/spancall/results_lean_v2_7/preds_hermes_test_adversarial.jsonl (hit=77, collapse_hits=0, e2e_parity=n/a)
predicted 200 rows (two_step) -> /content/spancall/results_lean_v2_7/preds_xlam_eval.jsonl (hit=141, collapse_hits=0, e2e_parity=468/506 (0.925))
predicted 200 rows (two_step) -> /content/spancall/results_lean_v2_7/preds_xlam_test_ood_schema.jsonl (hit=165, collapse_hits=0, e2e_parity=211/222 (0.950))

split                     system         n_F  exact_F   hit_F param_F1 exact_all
--------------------------------------------------------------------------------
hermes_test_ood_schema    b1a            113    0.053   0.000    0.000 

---
### 5b. Per-Corpus Evaluation (Hermes vs xLAM)

Splits every evaluation subset by `source_dataset` (Hermes vs xLAM) and reruns `scripts/eval_toolcall.py` per corpus, so Hermes-only and xLAM-only metrics are reported separately instead of blended. Runs after all subsets and predictions exist (eval split + all test splits).

In [16]:
import json, os, subprocess, sys

def _corpus_of(row):
    sd = str(row.get("source_dataset") or "")
    if sd.startswith("xlam"):
        return "xlam"
    if sd.startswith("hermes"):
        return "hermes"
    # Fallback: sample_id prefix ("xlam::..." vs hermes-style ids)
    return "xlam" if str(row.get("sample_id") or "").startswith("xlam") else "hermes"

_corpus_eval_pairs = []  # (split, corpus, label, metrics_path)

for split in TEST_SPLITS + [EVAL_SPLIT]:
    sub_path = SUBSET_EVAL_PATH if (split == EVAL_SPLIT and os.path.exists(SUBSET_EVAL_PATH)) \
        else os.path.join(RESULTS_DIR, f"subset_{split}.jsonl")
    if not os.path.exists(sub_path):
        continue
    with open(sub_path, encoding="utf-8") as fh:
        rows = [json.loads(l) for l in fh if l.strip()]
    by_corpus = {"hermes": [], "xlam": []}
    for r in rows:
        by_corpus[_corpus_of(r)].append(r)
    if split == EVAL_SPLIT:
        _pred_specs = [
            ("b1a", None),
            ("lean", os.path.join(RESULTS_DIR, "preds_eval.jsonl")),
            ("lean_twostep", os.path.join(RESULTS_DIR, "preds_eval_twostep.jsonl")),
        ]
    else:
        _pred_specs = [("b1a", None), ("lean", os.path.join(RESULTS_DIR, f"preds_{split}.jsonl"))]
    for corpus, crows in by_corpus.items():
        if not crows:
            continue
        csub = os.path.join(RESULTS_DIR, f"subset_{split}_{corpus}.jsonl")
        with open(csub, "w", encoding="utf-8") as fh:
            for r in crows:
                fh.write(json.dumps(r, default=str) + "\n")
        ids = {r["sample_id"] for r in crows}
        for label, pred_path in _pred_specs:
            cpred = None
            if pred_path is not None:
                if not os.path.exists(pred_path):
                    continue
                with open(pred_path, encoding="utf-8") as fh:
                    _pred_rows = [json.loads(l) for l in fh if l.strip()]
                _keep = [p for p in _pred_rows if p.get("sample_id") in ids]
                if not _keep:
                    continue
                cpred = os.path.join(RESULTS_DIR, f"preds_{split}_{corpus}_{label}.jsonl")
                with open(cpred, "w", encoding="utf-8") as fh:
                    for p in _keep:
                        fh.write(json.dumps(p, default=str) + "\n")
            out_json = os.path.join(RESULTS_DIR, f"toolcall_metrics_{split}_{corpus}_{label}.json")
            _cmd = [sys.executable, "scripts/eval_toolcall.py", "--test_data", csub]
            if label == "b1a":
                _cmd += ["--baseline", "b1a"]
            else:
                _cmd += ["--predictions", cpred]
            _cmd += ["--output", out_json]
            subprocess.run(_cmd, check=False)
            _corpus_eval_pairs.append((split, corpus, label, out_json))

print(f"\n=== Per-Corpus Evaluation (Hermes vs xLAM) ===")
print(f"{'split':<26}{'corpus':<9}{'system':<14}{'n_F':>6}{'exact_F':>9}{'hit_F':>8}{'param_F1':>9}{'exact_all':>10}")
print("-" * 90)
for split, corpus, label, out_json in _corpus_eval_pairs:
    m = _load_metrics(out_json)
    if not m:
        continue
    m_f = m.get("metrics_by_stratum", {}).get("F", {})
    print(f"{split:<26}{corpus:<9}{label:<14}{int(m_f.get('n', 0)):>6}"
          f"{m_f.get('exact_payload_match', 0.0):>9.3f}{m_f.get('hit_rate', 0.0):>8.3f}"
          f"{m_f.get('param_f1', 0.0):>9.3f}{m.get('exact_payload_match', 0.0):>10.3f}")


=== Per-Corpus Evaluation (Hermes vs xLAM) ===
split                     corpus   system           n_F  exact_F   hit_F param_F1 exact_all
------------------------------------------------------------------------------------------
hermes_test_ood_schema    hermes   b1a              113    0.053   0.000    0.000     0.038
hermes_test_ood_schema    hermes   lean             113    0.743   0.867    0.859     0.589
hermes_test_ood_para      hermes   b1a              177    0.051   0.011    0.015     0.051
hermes_test_ood_para      hermes   lean             177    0.734   0.831    0.858     0.734
hermes_test_adversarial   hermes   b1a                0    0.000   0.000    0.000     0.000
hermes_test_adversarial   hermes   lean               0    0.000   0.000    0.000     0.294
xlam_eval                 xlam     b1a              120    0.008   0.025    0.000     0.005
xlam_eval                 xlam     lean             120    0.617   0.817    0.686     0.400
xlam_test_ood_schema      xlam   

---
## 6. Final Run Summary Report

Aggregates the configuration, tensorized dataset counts, best training epoch, `hermes_eval` headline metrics, and all OOD/cross-corpus test results into a single summary printed to stdout and saved as `summary_report.json` / `summary_report.md`.

In [17]:
import json, os

def _count_jsonl(path: str) -> int:
    if not os.path.exists(path):
        return 0
    with open(path, encoding="utf-8") as fh:
        return sum(1 for line in fh if line.strip())

def _safe_round(v, ndigits: int = 4):
    if v is None:
        return None
    try:
        return round(float(v), ndigits)
    except (TypeError, ValueError):
        return None

_t2_n = _count_jsonl(os.path.join(TENSOR_DIR, "t2.jsonl"))
_dev_n = _count_jsonl(os.path.join(TENSOR_DIR, "dev.jsonl"))

_best_epoch = None
_all_epochs = []
_train_log_file = os.path.join(exp_root, "train_log.jsonl")
if os.path.exists(_train_log_file):
    with open(_train_log_file, encoding="utf-8") as _fh:
        _recs = [json.loads(l) for l in _fh if l.strip()]
    _evals = [r for r in _recs if r.get("event") == "epoch_eval"]
    _ends = {r.get("epoch"): r for r in _recs if r.get("event") == "epoch_end"}
    for _r in _evals:
        _ep = _r.get("epoch")
        _end = _ends.get(_ep, {})
        _entry = {
            "epoch": _ep,
            "span_acc": _safe_round(_r.get("span_acc")),
            "exact_params_only": _safe_round(_r.get("exact_params_only")),
            "exact_call": _safe_round(_r.get("exact_call")),
            "tool_acc": _safe_round(_r.get("tool_acc")),
            "eval_loss": _safe_round(_r.get("eval_loss")),
            "train_loss": _safe_round(_end.get("mean_loss")),
        }
        _all_epochs.append(_entry)
    if _all_epochs:
        _best_epoch = max(
            _all_epochs,
            key=lambda e: (e["exact_call"] if e["exact_call"] is not None else -1.0),
        )

_headline = {}
for _label, _fn in [
    ("B1a (regex)", f"toolcall_metrics_{EVAL_SPLIT}_b1a.json"),
    ("SpanCall Lean (single_pass)", f"toolcall_metrics_{EVAL_SPLIT}_lean.json"),
    ("SpanCall Lean (two_step)", f"toolcall_metrics_{EVAL_SPLIT}_lean_twostep.json"),
]:
    _m = _load_metrics(os.path.join(RESULTS_DIR, _fn))
    if not _m:
        continue
    _mf = _m.get("metrics_by_stratum", {}).get("F", {})
    _headline[_label] = {
        "n_F": int(_mf.get("n", 0)),
        "exact_F": _safe_round(_mf.get("exact_payload_match"), 3),
        "hit_F": _safe_round(_mf.get("hit_rate"), 3),
        "collapse_F": _safe_round(_mf.get("collapse_hit_rate"), 3),
        "param_F1_F": _safe_round(_mf.get("param_f1"), 3),
        "false_hit_on_A": _safe_round(_m.get("false_hit_rate_on_a"), 4),
        "exact_all": _safe_round(_m.get("exact_payload_match"), 3),
    }

_split_results = {}
for _sp in TEST_SPLITS + [EVAL_SPLIT]:
    _split_results[_sp] = {}
    for _lbl in ("b1a", "lean"):
        _m = _load_metrics(os.path.join(RESULTS_DIR, f"toolcall_metrics_{_sp}_{_lbl}.json"))
        if not _m:
            continue
        _mf = _m.get("metrics_by_stratum", {}).get("F", {})
        _split_results[_sp][_lbl] = {
            "n_F": int(_mf.get("n", 0)),
            "exact_F": _safe_round(_mf.get("exact_payload_match"), 3),
            "hit_F": _safe_round(_mf.get("hit_rate"), 3),
            "param_F1": _safe_round(_mf.get("param_f1"), 3),
            "false_hit_on_A": _safe_round(_m.get("false_hit_rate_on_a"), 4),
            "exact_all": _safe_round(_m.get("exact_payload_match"), 3),
        }

_summary = {
    "config": {
        "experiment_name": EXPERIMENT_NAME,
        "backbone": BACKBONE,
        "seed": SEED,
        "single_call_only": SINGLE_CALL_ONLY,
        "xlam_train_rows": XLAM_TRAIN_ROWS,
        "xlam_dev_rows": XLAM_DEV_ROWS,
        "max_calls": MAX_CALLS,
        "min_joint_margin": MIN_JOINT_MARGIN,
        "checkpoint_dir": CHECKPOINT_DIR,
    },
    "tensorized_rows": {"t2": _t2_n, "dev": _dev_n},
    "best_epoch": _best_epoch,
    "hermes_eval_headline": _headline,
    "split_results": _split_results,
}

_summary_json = os.path.join(RESULTS_DIR, "summary_report.json")
with open(_summary_json, "w", encoding="utf-8") as _fh:
    json.dump(_summary, _fh, indent=2)

_md_lines = [
    f"# SpanCall Lean v2.7 Run Summary ({EXPERIMENT_NAME})",
    "",
    "## Configuration & Data",
    f"- **Backbone**: `{BACKBONE}` | **Seed**: `{SEED}` | **Single-Call Only**: `{SINGLE_CALL_ONLY}`",
    f"- **xLAM Mix Caps**: `XLAM_TRAIN_ROWS={XLAM_TRAIN_ROWS}`, `XLAM_DEV_ROWS={XLAM_DEV_ROWS}`",
    f"- **Tensorized Rows**: `t2={_t2_n}`, `dev={_dev_n}`",
    f"- **Checkpoint**: `{CHECKPOINT_DIR}`",
    "",
    "## Best Dev Epoch",
]
if _best_epoch:
    _md_lines.append(
        f"- Epoch `{_best_epoch['epoch']}`: `exact_call={_best_epoch['exact_call']}`, "
        f"`tool_acc={_best_epoch['tool_acc']}`, `span_acc={_best_epoch['span_acc']}`, "
        f"`exact_params_only={_best_epoch['exact_params_only']}`, `eval_loss={_best_epoch['eval_loss']}`"
    )
else:
    _md_lines.append("- No training log found.")

_md_lines.extend([
    "",
    f"## Headline Comparison ({EVAL_SPLIT})",
    "| System | n_F | exact_F | hit_F | collapse_F | param_F1_F | false_hit_on_A | exact_all |",
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |",
])
for _sys_name, _vals in _headline.items():
    _md_lines.append(
        f"| {_sys_name} | {_vals['n_F']} | {_vals['exact_F']} | {_vals['hit_F']} | "
        f"{_vals['collapse_F']} | {_vals['param_F1_F']} | {_vals['false_hit_on_A']} | {_vals['exact_all']} |"
    )

_md_lines.extend([
    "",
    "## All Test Splits (two_step Lean vs B1a)",
    "| Split | System | n_F | exact_F | hit_F | param_F1 | false_hit_on_A | exact_all |",
    "| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |",
])
for _sp, _by_sys in _split_results.items():
    for _lbl, _vals in _by_sys.items():
        _md_lines.append(
            f"| {_sp} | {_lbl} | {_vals['n_F']} | {_vals['exact_F']} | {_vals['hit_F']} | "
            f"{_vals['param_F1']} | {_vals['false_hit_on_A']} | {_vals['exact_all']} |"
        )

_summary_md_text = "\n".join(_md_lines) + "\n"
_summary_md = os.path.join(RESULTS_DIR, "summary_report.md")
with open(_summary_md, "w", encoding="utf-8") as _fh:
    _fh.write(_summary_md_text)

print(_summary_md_text)
print(f"Saved summary -> {_summary_json} & {_summary_md}")

# SpanCall Lean v2.7 Run Summary (spancall_lean_modernbert)

## Configuration & Data
- **Backbone**: `answerdotai/ModernBERT-large` | **Seed**: `42` | **Single-Call Only**: `True`
- **xLAM Mix Caps**: `XLAM_TRAIN_ROWS=15000`, `XLAM_DEV_ROWS=1000`
- **Tensorized Rows**: `t2=20779`, `dev=850`
- **Checkpoint**: `/content/spancall/results_lean_v2_7/spancall_lean_modernbert_seed42/best_checkpoint`

## Best Dev Epoch
- Epoch `3`: `exact_call=0.78`, `tool_acc=0.9953`, `span_acc=0.8306`, `exact_params_only=0.7824`, `eval_loss=0.6126`

## Headline Comparison (hermes_eval)
| System | n_F | exact_F | hit_F | collapse_F | param_F1_F | false_hit_on_A | exact_all |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| B1a (regex) | 259 | 0.019 | 0.023 | 0.0 | 0.021 | 0.0192 | 0.012 |
| SpanCall Lean (single_pass) | 259 | 0.649 | 0.907 | 0.0 | 0.818 | 0.3462 | 0.441 |
| SpanCall Lean (two_step) | 259 | 0.637 | 0.896 | 0.0 | 0.814 | 0.3526 | 0.434 |

## All Test Splits (two_step Lean vs B1a)
| Sp